# Sionna 2.0 — 915.95 MHz Simulation (Nottingham)

**Frequency:** 915.95 MHz · **RX:** 1200 sequential from CSV · **Terrain:** flat · **Backend:** PyTorch (Sionna 2.0)

**CSV:** nottingham915.csv  ·  TX EIRP=50.3 dBm  ·  RX system gain=−7.8 dB

Adapted from `sionna019_900mhz_simulation.ipynb` using the Sionna 2.0 API from `sionna2_main_simulation.ipynb`.

## Cell 0 — Environment Setup

Sionna 2.0 uses PyTorch — no TensorFlow, no Mitsuba variant setup needed.

In [ ]:
import os, sys, json, csv, time, warnings, glob, re
import xml.etree.ElementTree as ET
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 100})
import matplotlib.pyplot as plt
from scipy.constants import speed_of_light as C
from pyproj import Transformer
from datetime import datetime
import math

# Sionna 2.0 — PyTorch backend, no TensorFlow, no Mitsuba variant setup
import torch
import sionna
import sionna.rt as rt
from sionna.rt import load_scene, RadioMaterial, PlanarArray, Transmitter, Receiver, PathSolver

_HAS_RIO = False
try:
    import rasterio as rio
    _HAS_RIO = True
    print('rasterio: OK')
except ImportError:
    print('rasterio: NOT available — flat terrain fallback will be used')

_HAS_OSM = False
try:
    import osmnx as ox
    _HAS_OSM = True
    print('osmnx   : OK')
except ImportError:
    print('osmnx   : NOT available')

def _safe(v):
    """Convert tensor or numeric value to Python float safely."""
    if hasattr(v, 'item'):  return float(v.item())
    if hasattr(v, 'numpy'): return float(v.numpy())
    return float(v)

print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'Sionna  : {sionna.__version__}')

## Cell 1 — Configuration (900 MHz / Nottingham)

In [ ]:
# ============================================================
# CELL 1 — CONFIGURATION
# ============================================================
# Edit SCENARIO block only. All paths are auto-derived.
# To port to a new campaign: change the SCENARIO section.
# ============================================================
import os

# ╔══════════════════════════════════════════════════════════╗
# ║                  SCENARIO CONFIGURATION                  ║
# ║           Edit this block for each new campaign          ║
# ╚══════════════════════════════════════════════════════════╝

SCENARIO_NAME   = 'nottingham_ofcom2018_915mhz'  # used for output filenames

# ── City / coordinate system ──────────────────────────────
CITY_NAME    = 'Nottingham'
UTM_EPSG     = 32630          # UTM zone 30N (UK); change per city

# ── Scene bbox (WGS84) ───────────────────────────────────
SCENE_WEST   = -1.260093
SCENE_EAST   = -1.129307
SCENE_SOUTH  =  52.945798
SCENE_NORTH  =  52.998702

# ── TX parameters ────────────────────────────────────────
TX_LON              = -1.2559
TX_LAT              =  52.9863
TX_AGL_M            = 23.0           # m AGL
TX_CONDUCTED_DBM    = 49.0           # dBm
TX_ANTENNA_GAIN_DBI =  1.3           # dBi (collinear omni)
EIRP_DBM            = TX_CONDUCTED_DBM + TX_ANTENNA_GAIN_DBI

# ── Antenna pattern ───────────────────────────────────────
# 'iso'   = isotropic (0 dBi)
# 'donut' = half-wave dipole (~2.15 dBi, nulls at zenith/nadir)
ANTENNA_PATTERN = 'donut'

# ── RX parameters ────────────────────────────────────────
RX_AGL_M           =  1.5            # m AGL
RX_EXTRA_GAIN_DB   = -7.8            # dB (cable + filter losses)
SITE_CORRECTION_DB =  0.0            # dB (post-hoc calibration offset)
NOISE_FLOOR_DBM    = -124.0          # dBm

# ── RX selection ─────────────────────────────────────────
NUM_RX       = 1200                  # number of receivers to use

# ── Frequency ────────────────────────────────────────────
FREQUENCY_HZ = 915.95e6              # Hz

# ── Terrain ──────────────────────────────────────────────
FLAT_TERRAIN = True                  # False → use DEM GeoTIFF (run CELL 2b first)

# ── Ray tracing ──────────────────────────────────────────
MAX_DEPTH        = 3                 # max reflection/diffraction bounces
NUM_SAMPLES_PS   = 1_000_000         # rays per PathSolver call
BATCH_SIZE       = 5                 # receivers per batch

# ╔══════════════════════════════════════════════════════════╗
# ║                    PATH CONFIGURATION                    ║
# ║   Auto-derived from SCENARIO_NAME — no edits needed      ║
# ╚══════════════════════════════════════════════════════════╝

# Root: ~/sionna_rt/<scenario_name>/
# Override any path below if your folder layout differs.
_ROOT           = os.path.expanduser(f'~/sionna_rt/{SCENARIO_NAME}')

BASE_DIR        = _ROOT
SCENE_DIR       = os.path.join(_ROOT, 'scene')
BLENDER_DIR     = os.path.join(_ROOT, 'blender')
OUT_DIR         = os.path.join(_ROOT, 'results')
DEM_TIFF        = os.path.join(_ROOT, 'dem.tif')

SCENE_XML       = os.path.join(BLENDER_DIR, 'scene_sionna2.xml')
OFCOM_RAW_CSV   = os.path.join(_ROOT, 'measurements_raw.csv')
RX_CSV          = os.path.join(SCENE_DIR, 'receiver_locations.csv')
MEASUREMENT_CSV = os.path.join(SCENE_DIR, 'measurements_with_pathloss.csv')

os.makedirs(OUT_DIR, exist_ok=True)


# ╔══════════════════════════════════════════════════════════╗
# ║              FREQUENCY PRESETS — SWAP BLOCK              ║
# ║   Uncomment one block to switch frequency campaign       ║
# ╚══════════════════════════════════════════════════════════╝

# ── PRESET A: 915.95 MHz (Ofcom 2018 — current) ──────────
# SCENARIO_NAME = 'nottingham_ofcom2018_915mhz'
# FREQUENCY_HZ  = 915.95e6
# MAX_DEPTH     = 3
# MATERIAL_FREQ = '900mhz'

# ── PRESET B: 3.602 GHz ──────────────────────────────────
# SCENARIO_NAME = 'nottingham_ofcom2018_3602mhz'
# FREQUENCY_HZ  = 3.602e9
# MAX_DEPTH     = 2     # walls absorb more → fewer meaningful bounces
# MATERIAL_FREQ = '3600mhz'
# TX_CONDUCTED_DBM    = 49.0
# TX_ANTENNA_GAIN_DBI =  1.3
# EIRP_DBM            = TX_CONDUCTED_DBM + TX_ANTENNA_GAIN_DBI
# RX_EXTRA_GAIN_DB    = -7.8
# Note: re-run CELL 4A after switching — sigma values differ at 3.6 GHz

# ── Optional: override paths if data lives elsewhere ─────
# Uncomment and set absolute paths to override auto-derived paths above.
# BASE_DIR      = '/path/to/your/data'
# SCENE_XML     = '/path/to/scene_sionna2.xml'
# OFCOM_RAW_CSV = '/path/to/measurements.csv'

# ── RSSI from path gain (Sionna 2.0) ─────────────────────
def rssi_from_path_gain(path_gain_linear):
    pg = np.asarray(path_gain_linear, dtype=float)
    pl = np.where(pg > 0, -10.0 * np.log10(np.maximum(pg, 1e-30)), np.nan)
    return TX_CONDUCTED_DBM - pl + RX_EXTRA_GAIN_DB + SITE_CORRECTION_DB

print(f'Scenario         : {SCENARIO_NAME}')
print(f'Frequency        : {FREQUENCY_HZ/1e6:.2f} MHz')
print(f'TX               : lon={TX_LON}  lat={TX_LAT}  AGL={TX_AGL_M}m')
print(f'TX conducted     : {TX_CONDUCTED_DBM} dBm  antenna={TX_ANTENNA_GAIN_DBI} dBi  EIRP={EIRP_DBM:.1f} dBm')
print(f'Antenna pattern  : {ANTENNA_PATTERN}')
print(f'RX system gain   : {RX_EXTRA_GAIN_DB} dB  |  Site corr: {SITE_CORRECTION_DB} dB')
print(f'Noise floor      : {NOISE_FLOOR_DBM} dBm')
print(f'NUM_RX           : {NUM_RX}  |  MAX_DEPTH: {MAX_DEPTH}  |  BATCH: {BATCH_SIZE}')
print(f'Flat terrain     : {FLAT_TERRAIN}')
print(f'Base dir         : {BASE_DIR}')
print(f'Scene XML        : {SCENE_XML}')
print(f'Raw CSV          : {OFCOM_RAW_CSV}')
print()
# ── Warn if key paths are missing ────────────────────────
for _label, _path in [('SCENE_XML', SCENE_XML), ('OFCOM_RAW_CSV', OFCOM_RAW_CSV)]:
    if not os.path.exists(_path):
        print(f'  WARNING: {_label} not found: {_path}')
        print(f'           → set correct path in the OPTIONAL OVERRIDE block above')


## GPS → Scene Coordinate Transform — Method & Reference

All geographic coordinates in this notebook are converted to Sionna scene-local metres using a two-stage pipeline consistent with the Sionna RT community approach (geo2sigmap, sionna-large-radio-maps).

**Stage 1 — WGS84 to UTM.** The raw GPS coordinates (EPSG:4326, decimal degrees) are projected to UTM Zone 30N (EPSG:32630) using pyproj `Transformer.from_crs` with `always_xy=True`. The `always_xy` flag enforces `(longitude, latitude)` → `(easting, northing)` axis order regardless of the CRS convention, preventing the silent axis swap that is a common source of multi-kilometre placement errors.

**Stage 2 — UTM to scene-local.** The scene origin is defined as the arithmetic midpoint of the scene bounding box `((SCENE_WEST+SCENE_EAST)/2, (SCENE_SOUTH+SCENE_NORTH)/2)`. Every antenna position is expressed as the signed metre offset from this origin `(easting − origin_easting, northing − origin_northing)`. This matches the origin used by the OSM scene builder.

**Stage 3 — Terrain height.** Antenna height is expressed as height Above Ground Level (AGL). Ground elevation at each antenna site is sampled by bilinear interpolation from a merged SRTM/LiDAR DEM (GeoTIFF, WGS84, 1–30 m resolution). When the DEM file is absent the notebook falls back to flat terrain (z=0 everywhere).

**Reference:** pyproj `Transformer` (RFC 7946, EPSG:4326 ↔ EPSG:32630); Sionna RT positions are pure Cartesian metres — confirmed in NVlabs/sionna Discussion #154.

## Cell 2 — Coordinate Utilities (GPS → UTM → Local, DEM Terrain)

In [ ]:
# ── Coordinate transformers (pyproj, WGS84 ↔ UTM 30N) ──────────────────────
# always_xy=True enforces (lon, lat) / (easting, northing) order
gps_to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
utm_to_gps = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)

# Scene centre in UTM (origin of local coordinate system)
center_lon = (SCENE_WEST + SCENE_EAST)  / 2
center_lat = (SCENE_SOUTH + SCENE_NORTH) / 2
utm_center_x, utm_center_y = gps_to_utm.transform(center_lon, center_lat)

def gps_to_local(lon, lat, height=0.0):
    """WGS84 (lon, lat) → scene-local (x, y, z) in metres.
    Origin = scene bbox centre. X = east, Y = north, Z = up."""
    ux, uy = gps_to_utm.transform(lon, lat)
    return float(ux - utm_center_x), float(uy - utm_center_y), float(height)

def local_to_gps(x, y):
    """Scene-local (x, y) → WGS84 (lon, lat)."""
    lon, lat = utm_to_gps.transform(x + utm_center_x, y + utm_center_y)
    return float(lon), float(lat)

# ── DEM terrain elevation ─────────────────────────────────────────────────────
dem_data = dem_nodata = dem_tf = dem_crs = None
if _HAS_RIO and not FLAT_TERRAIN and os.path.exists(DEM_TIFF):
    _src      = rio.open(DEM_TIFF)
    dem_data  = _src.read(1).astype(np.float32)
    dem_nodata= _src.nodata
    dem_tf    = _src.transform
    dem_crs   = str(_src.crs)
    print(f'DEM loaded : {dem_data.shape}  nodata={dem_nodata}  CRS={dem_crs}')
else:
    if FLAT_TERRAIN:
        print('DEM        : FLAT_TERRAIN=True — z=0 everywhere')
    else:
        print(f'DEM        : not found at {DEM_TIFF} — falling back to flat terrain')

def get_dem_elevation(local_x, local_y):
    """Return absolute terrain elevation (m ASL) at scene-local (x, y)."""
    if dem_data is None: return 0.0
    utm_x = float(local_x) + utm_center_x
    utm_y = float(local_y) + utm_center_y
    # DEM is in WGS84 lon/lat — convert UTM → lon/lat for pixel lookup
    lon, lat = utm_to_gps.transform(utm_x, utm_y)
    col_f, row_f = ~dem_tf * (lon, lat)
    r, c = int(np.floor(row_f)), int(np.floor(col_f))
    H, W = dem_data.shape
    if 0 <= r < H-1 and 0 <= c < W-1:
        dr, dc = row_f - r, col_f - c
        z = ((1-dr)*(1-dc)*dem_data[r,c]   + (1-dr)*dc*dem_data[r,c+1] +
              dr*(1-dc)*dem_data[r+1,c]    + dr*dc*dem_data[r+1,c+1])
        if dem_nodata is None or not np.isclose(float(z), dem_nodata):
            return float(z)
    return 0.0

# Scene z=0 corresponds to DEM elevation at scene centre.
_SCENE_ORIGIN_ELEV = get_dem_elevation(0.0, 0.0)

def terrain_z(local_x, local_y):
    """Return scene-local Z of ground at (x,y). 0.0 if flat terrain."""
    if FLAT_TERRAIN or dem_data is None: return 0.0
    return get_dem_elevation(local_x, local_y) - _SCENE_ORIGIN_ELEV

print(f'Scene centre  : lon={center_lon:.6f}  lat={center_lat:.6f}')
print(f'UTM centre    : ({utm_center_x:.1f}, {utm_center_y:.1f})')
print(f'Scene origin elevation: {_SCENE_ORIGIN_ELEV:.1f} m ASL')
print(f'gps_to_local(TX): {gps_to_local(TX_LON, TX_LAT)}')

## Cell 3 — Load Scene (Sionna 2.0)

In [ ]:
# ====================================================================
# CELL 4A — ASSIGN REALISTIC MATERIALS (FULL COVERAGE)
# ====================================================================
import numpy as np

print("=" * 70)
print("ASSIGNING REALISTIC MATERIAL PROPERTIES (AUTO-MATCH)")
print("=" * 70)


# ── Frequency-aware material properties ──────────────────────────────────────
# ITU-R P.2040-2 values differ by frequency — select correct block
_freq_ghz = FREQUENCY_HZ / 1e9

if _freq_ghz < 2.0:   # 900 MHz band
    MATERIAL_PROPS = {
        "concrete":    {"er": 5.31,  "sigma": 0.092,  "scatter": 0.20},
        "brick":       {"er": 3.75,  "sigma": 0.038,  "scatter": 0.25},
        "glass":       {"er": 6.27,  "sigma": 0.000,  "scatter": 0.08},
        "metal":       {"er": 1.00,  "sigma": 1e7,    "scatter": 0.05},
        "wood":        {"er": 1.99,  "sigma": 0.000,  "scatter": 0.30},
        "asphalt":     {"er": 2.56,  "sigma": 0.000,  "scatter": 0.30},
        "wet_ground":  {"er": 30.0,  "sigma": 0.020,  "scatter": 0.40},
        "very_dry":    {"er": 2.80,  "sigma": 0.000,  "scatter": 0.30},
        "medium_dry":  {"er": 4.00,  "sigma": 0.001,  "scatter": 0.35},
        "ground":      {"er": 4.00,  "sigma": 0.001,  "scatter": 0.35},
        "vegetation":  {"er": 1.50,  "sigma": 0.000,  "scatter": 0.75},
        "water":       {"er": 80.0,  "sigma": 0.020,  "scatter": 0.02},
    }
    print(f"Material set: 900 MHz band ({_freq_ghz:.3f} GHz)")

else:                  # 3.6 GHz band
    MATERIAL_PROPS = {
        "concrete":    {"er": 5.31,  "sigma": 0.310,  "scatter": 0.20},
        "brick":       {"er": 3.75,  "sigma": 0.100,  "scatter": 0.25},
        "glass":       {"er": 6.27,  "sigma": 0.018,  "scatter": 0.10},
        "metal":       {"er": 1.00,  "sigma": 1e7,    "scatter": 0.05},
        "wood":        {"er": 1.99,  "sigma": 0.047,  "scatter": 0.30},
        "asphalt":     {"er": 2.56,  "sigma": 0.020,  "scatter": 0.30},
        "wet_ground":  {"er": 20.0,  "sigma": 0.110,  "scatter": 0.40},
        "very_dry":    {"er": 2.80,  "sigma": 0.001,  "scatter": 0.30},
        "medium_dry":  {"er": 4.00,  "sigma": 0.008,  "scatter": 0.35},
        "ground":      {"er": 4.00,  "sigma": 0.008,  "scatter": 0.35},
        "vegetation":  {"er": 1.50,  "sigma": 0.060,  "scatter": 0.75},
        "water":       {"er": 80.0,  "sigma": 0.180,  "scatter": 0.02},
    }
    print(f"Material set: 3.6 GHz band ({_freq_ghz:.3f} GHz)")


# ── List existing materials ───────────────────────────────────────────────────
existing_materials = list(scene.radio_materials.keys())
print("Existing materials:")
for name in existing_materials:
    print(f"  {name}")
print()

# ── Diagnostic: find actual scattering attribute names ───────────────────────
print("Scattering attribute scan (first material):")
_first_mat = scene.radio_materials[existing_materials[0]]
for _attr in dir(_first_mat):
    if 'scat' in _attr.lower():
        _val = getattr(_first_mat, _attr, 'N/A')
        print(f"  {_attr} = {_val}")
print()

# ── Assign properties ─────────────────────────────────────────────────────────
updated_count = 0
for mat_name in existing_materials:
    name_lower = mat_name.lower()
    matched_type = None
    for mat_type in MATERIAL_PROPS:
        if mat_type in name_lower or name_lower in mat_type:
            matched_type = mat_type
            break
    if matched_type is None:
        print(f"  ✗ {mat_name:30s} → no match")
        continue

    props = MATERIAL_PROPS[matched_type]
    mat   = scene.radio_materials[mat_name]
    _s    = props["scatter"]

    # Set EM properties (known writable)
    mat.relative_permittivity = props["er"]
    mat.conductivity          = props["sigma"]

    # Try ALL known scattering attribute names in Sionna 2.0
    _scat_set = False
    for _attr in ('scattering_coefficient',
                  '_scattering_coefficient',
                  'scattering_pattern_coefficient',
                  'lambda_',
                  'scat_coeff',
                  '_scat_coeff'):
        if hasattr(mat, _attr):
            try:
                _cur = getattr(mat, _attr)
                if hasattr(_cur, 'data'):        # torch.nn.Parameter
                    _cur.data.fill_(_s)
                    _scat_set = True
                elif hasattr(_cur, 'assign'):    # tf.Variable
                    _cur.assign(_s)
                    _scat_set = True
                else:
                    setattr(mat, _attr, _s)
                    _scat_set = True
                if _scat_set:
                    break
            except Exception as _e:
                pass

    updated_count += 1
    print(f"  ✓ {mat_name:30s} → {matched_type:12s}  "
          f"ε={props['er']:.2f}  σ={props['sigma']:.4f}  "
          f"scatter={'SET' if _scat_set else 'FAILED'}")

print(f"\nUpdated {updated_count}/{len(existing_materials)} materials.")

# ── Verify ────────────────────────────────────────────────────────────────────
print("\n[VERIFICATION]")
for mat_name in existing_materials:
    mat = scene.radio_materials[mat_name]
    _sv = None
    for _a in ('scattering_coefficient', '_scattering_coefficient'):
        if hasattr(mat, _a):
            _v = getattr(mat, _a)
            try:
                _sv = float(_v.item()) if hasattr(_v, 'item') else (
                      float(_v.numpy().flat[0]) if hasattr(_v, 'numpy') else float(_v))
            except Exception:
                _sv = str(_v)[:30]
            break
    _er = _safe(mat.relative_permittivity)
    _sg = _safe(mat.conductivity)
    print(f"  {mat_name:30s}  ε={_er:.2f}  σ={_sg:.4f}  scatter={_sv}")

print("\n✅ Done.")


In [ ]:
# ====================================================================
# CELL 4A — ASSIGN REALISTIC MATERIALS (FULL COVERAGE)
# ====================================================================
import numpy as np

print("=" * 70)
print("ASSIGNING REALISTIC MATERIAL PROPERTIES (AUTO-MATCH)")
print("=" * 70)

# ── Material property database ────────────────────────────────────────────────
MATERIAL_PROPS = {
    # Building materials
    "concrete":         {"er": 5.31,  "sigma": 0.107,  "thickness": 0.30,  "scatter": 0.15},
    "brick":            {"er": 3.91,  "sigma": 0.0238, "thickness": 0.25,  "scatter": 0.12},
    "glass":            {"er": 6.27,  "sigma": 0.012,  "thickness": 0.005, "scatter": 0.08},
    "metal":            {"er": 1.00,  "sigma": 1e7,    "thickness": 0.01,  "scatter": 0.05},
    "wood":             {"er": 1.99,  "sigma": 0.0047, "thickness": 0.10,  "scatter": 0.25},
    # Ground & paving
    "asphalt":          {"er": 3.00,  "sigma": 0.010,  "thickness": 0.15,  "scatter": 0.35},
    "dry_ground":       {"er": 15.0,  "sigma": 0.035,  "thickness": 0.10,  "scatter": 0.18},
    "wet_ground":       {"er": 30.0,  "sigma": 0.150,  "thickness": 0.10,  "scatter": 0.20},
    "ground":           {"er": 15.0,  "sigma": 0.035,  "thickness": 0.10,  "scatter": 0.18},
    # Vegetation & water
    "vegetation":       {"er": 1.30,  "sigma": 0.001,  "thickness": 1.0,   "scatter": 0.75},
    "foliage":          {"er": 1.30,  "sigma": 0.001,  "thickness": 1.0,   "scatter": 0.75},
    "water":            {"er": 81.0,  "sigma": 0.500,  "thickness": 0.10,  "scatter": 0.02},
}

# ── List existing materials ───────────────────────────────────────────────────
existing_materials = list(scene.radio_materials.keys())
print("Existing materials in scene:")
for name in existing_materials:
    print(f"  {name}")
print()

# ── Assign by partial match (case-insensitive) ────────────────────────────────
updated_count = 0
for mat_name in existing_materials:
    name_lower = mat_name.lower()
    matched_type = None
    for mat_type in MATERIAL_PROPS.keys():
        if mat_type in name_lower or name_lower in mat_type:
            matched_type = mat_type
            break
    if matched_type is not None:
        props = MATERIAL_PROPS[matched_type]
        mat = scene.radio_materials[mat_name]
        mat.relative_permittivity   = props["er"]
        mat.conductivity            = props["sigma"]
        mat.thickness               = props["thickness"]
        mat.scattering_coefficient  = props["scatter"]
        updated_count += 1
        print(f"  ✓ {mat_name:30s} → {matched_type:12s}  "
              f"ε={props['er']:.2f}  σ={props['sigma']:.4f}  "
              f"thick={props['thickness']:.3f}m  scatter={props['scatter']:.2f}")
    else:
        print(f"  ✗ {mat_name:30s} → no match (unchanged)")

print(f"\nUpdated {updated_count}/{len(existing_materials)} materials.")

# ── Verify ────────────────────────────────────────────────────────────────────
print("\n[VERIFICATION]")
for mat_name in existing_materials:
    mat = scene.radio_materials[mat_name]
    thick_val   = _safe(mat.thickness)           if hasattr(mat, "thickness")           else "N/A"
    scatter_val = _safe(mat.scattering_coefficient) if hasattr(mat, "scattering_coefficient") else "N/A"
    er_val      = _safe(mat.relative_permittivity)  if hasattr(mat, "relative_permittivity")  else "N/A"
    sigma_val   = _safe(mat.conductivity)            if hasattr(mat, "conductivity")            else "N/A"
    print(f"  {mat_name:30s}  ε={er_val}  σ={sigma_val}  thick={thick_val}  scatter={scatter_val}")

print("\n✅ Material properties assigned.")


## Cell 4 — Place TX

In [ ]:
# Remove previous TX
for _n in list(scene.transmitters.keys()):
    scene.remove(_n)

tx_x, tx_y, _ = gps_to_local(TX_LON, TX_LAT)
tx_z = terrain_z(tx_x, tx_y) + TX_AGL_M  # ground + AGL

# ── Antenna pattern ───────────────────────────────────────────────────────────
# 'iso'   = isotropic (0 dBi, uniform sphere) — baseline
# 'donut' = vertical half-wave dipole (~2.15 dBi, null at zenith/nadir)
#           matches the collinear omni antenna used in the Ofcom campaign
if ANTENNA_PATTERN == 'donut':
    _tx_ant = PlanarArray(num_rows=1, num_cols=1,
                          vertical_spacing=0.5,
                          horizontal_spacing=0.5,
                          pattern="dipole",
                          polarization="V")
    _ant_desc = "half-wave dipole (donut, ~2.15 dBi)"
else:
    _tx_ant = PlanarArray(num_rows=1, num_cols=1,
                          vertical_spacing=0.5,
                          horizontal_spacing=0.5,
                          pattern="iso",
                          polarization="V")
    _ant_desc = "isotropic (0 dBi)"

tx = Transmitter(name='tx0',
                 position=[tx_x, tx_y, tx_z],
                 orientation=[0.0, 0.0, 0.0])
tx.antenna = _tx_ant
scene.add(tx)

print(f'TX placed:')
print(f'  GPS      : lon={TX_LON}  lat={TX_LAT}')
print(f'  Local    : ({tx_x:.1f}, {tx_y:.1f}, {tx_z:.1f}) m')
print(f'  AGL      : {TX_AGL_M} m  (terrain_z={tx_z - TX_AGL_M:.1f} m)')
print(f'  Antenna  : {_ant_desc}  [ANTENNA_PATTERN=\'{ANTENNA_PATTERN}\']')


## Cell 5 — Extract RX from Ofcom CSV

In [ ]:
import csv as _csv_mod

print('=' * 60)
print('CELL 5 — RX EXTRACTION (first 1200 sequential from CSV)')
print('=' * 60)

if not os.path.exists(OFCOM_RAW_CSV):
    raise FileNotFoundError(f'CSV not found: {OFCOM_RAW_CSV}')

# ── Auto-detect header row ────────────────────────────────────────────────────
# Scan up to the first 40 lines for the row that contains both
# 'Latitude' and 'Longitude' — works regardless of metadata lines before data
_hdr_idx = None
with open(OFCOM_RAW_CSV, 'r', encoding='utf-8', errors='replace') as _f:
    for _i, _line in enumerate(_f):
        if 'Latitude' in _line and 'Longitude' in _line:
            _hdr_idx = _i
            break
        if _i > 40:
            break

if _hdr_idx is None:
    raise ValueError(
        f'Could not find header row in {OFCOM_RAW_CSV}\n'
        'Expected a line containing both "Latitude" and "Longitude" in the first 40 lines.')

print(f'Header at line {_hdr_idx + 1}  (0-based index {_hdr_idx})')
_df_raw = pd.read_csv(OFCOM_RAW_CSV, skiprows=_hdr_idx, low_memory=False)
print(f'Columns: {list(_df_raw.columns)}')
print(f'Total rows: {len(_df_raw)}')

# ── Column mapping — flexible: match by keyword ───────────────────────────────
def _find_col(df, *keywords):
    """Return first column name that contains all keywords (case-insensitive)."""
    for col in df.columns:
        c = col.strip().lower()
        if all(k.lower() in c for k in keywords):
            return col
    return None

_lat_col  = _find_col(_df_raw, 'latitude')
_lon_col  = _find_col(_df_raw, 'longitude')
_rssi_col = _find_col(_df_raw, 'measurement') or _find_col(_df_raw, 'dBm') or _find_col(_df_raw, 'dbm')

if not _lat_col:
    raise KeyError(f'No Latitude column found. Available: {list(_df_raw.columns)}')
if not _lon_col:
    raise KeyError(f'No Longitude column found. Available: {list(_df_raw.columns)}')
if not _rssi_col:
    raise KeyError(f'No RSSI/measurement column found. Available: {list(_df_raw.columns)}')

print(f'Lat  col : {_lat_col!r}')
print(f'Lon  col : {_lon_col!r}')
print(f'RSSI col : {_rssi_col!r}')

# Drop rows with non-numeric values in key columns
for _c in [_lat_col, _lon_col, _rssi_col]:
    _df_raw[_c] = pd.to_numeric(_df_raw[_c], errors='coerce')
_df_raw = _df_raw.dropna(subset=[_lat_col, _lon_col, _rssi_col]).reset_index(drop=True)
print(f'Rows after numeric filter: {len(_df_raw)}')

# Take first NUM_RX rows in CSV order (sequential as recorded)
_sel = _df_raw.head(NUM_RX).copy()

# Distance from TX (for info only)
_dlon_m = 111000.0 * math.cos(math.radians(TX_LAT))
_dlat_m = 111000.0
_sel['_dist_km'] = (
    ((_sel[_lat_col] - TX_LAT) * _dlat_m)**2 +
    ((_sel[_lon_col] - TX_LON) * _dlon_m)**2
)**0.5 / 1000.0

print(f'\nSelected  : {len(_sel)} receivers (rows 1–{len(_sel)}, sequential CSV order)')
print(f'Dist range: {_sel["_dist_km"].min():.3f} – {_sel["_dist_km"].max():.3f} km')
print(f'RSSI range: {_sel[_rssi_col].min():.1f} – {_sel[_rssi_col].max():.1f} dBm')

# ── Write receiver_locations.csv ──────────────────────────────────────────────
os.makedirs(os.path.dirname(RX_CSV), exist_ok=True)
with open(RX_CSV, 'w', newline='') as _f:
    _w = _csv_mod.writer(_f)
    _w.writerow(['name', 'lon', 'lat', 'height'])
    for _idx, _row in _sel.iterrows():
        _w.writerow([f'RX_{_idx:06d}',
                     f'{float(_row[_lon_col]):.6f}',
                     f'{float(_row[_lat_col]):.6f}',
                     RX_AGL_M])
print(f'\nWritten : {RX_CSV}  ({len(_sel)} rows)')

# ── Write measurements_with_pathloss.csv ──────────────────────────────────────
with open(MEASUREMENT_CSV, 'w', newline='') as _f:
    _w = _csv_mod.writer(_f)
    _w.writerow(['name', 'lon', 'lat', 'local_measurement_dBm', 'path_loss_dB'])
    for _idx, _row in _sel.iterrows():
        _rssi = float(_row[_rssi_col])
        _pl   = EIRP_DBM - _rssi + RX_EXTRA_GAIN_DB
        _w.writerow([f'RX_{_idx:06d}',
                     f'{float(_row[_lon_col]):.6f}',
                     f'{float(_row[_lat_col]):.6f}',
                     f'{_rssi:.2f}',
                     f'{_pl:.2f}'])
print(f'Written : {MEASUREMENT_CSV}  ({len(_sel)} rows)')

## Cell 6 — Place Receivers

In [ ]:
df_rx = pd.read_csv(RX_CSV)
for nm in list(scene.receivers.keys()):
    scene.remove(nm)

receivers = []
for _, row in df_rx.iterrows():
    lx, ly, _ = gps_to_local(float(row['lon']), float(row['lat']))
    lz = terrain_z(lx, ly) + RX_AGL_M  # ground + AGL
    rx = Receiver(name=row['name'], position=[lx, ly, lz])
    scene.add(rx)
    receivers.append(rx)

print(f'Placed {len(receivers)} receivers at z={RX_AGL_M}m (flat terrain={FLAT_TERRAIN})')

In [ ]:
# ====================================================================
# CELL 5b — EXPORT RECEIVER COORDINATES
# ====================================================================
# Writes a CSV with, for each receiver:
#   id, lat, lon, x_m, y_m, z_m, dist_from_tx_m
# ====================================================================
import numpy as np, pandas as pd, os

print('=' * 60)
print('CELL 5b — EXPORT RECEIVER COORDINATES')
print('=' * 60)

_safe_e = lambda v: float(v.item()) if hasattr(v, 'item') else float(v)

# TX Cartesian position
_tx_e   = list(scene.transmitters.values())[0]
_tx_xyz = np.array([_safe_e(_tx_e.position[0]),
                    _safe_e(_tx_e.position[1]),
                    _safe_e(_tx_e.position[2])])

# Re-load original lat/lon from receiver_locations CSV (written by CELL 5)
_df_locs = pd.read_csv(RX_CSV)   # columns: name, lon, lat, height

rows = []
for _rx in receivers:
    _x = _safe_e(_rx.position[0])
    _y = _safe_e(_rx.position[1])
    _z = _safe_e(_rx.position[2])
    _d = float(np.linalg.norm([_x - _tx_xyz[0], _y - _tx_xyz[1]]))

    # Match lat/lon from CSV by name
    _match = _df_locs[_df_locs['name'] == _rx.name]
    if len(_match):
        _lat = float(_match.iloc[0]['lat'])
        _lon = float(_match.iloc[0]['lon'])
    else:
        _lat = _lon = float('nan')

    rows.append({
        'id'           : _rx.name,
        'lat'          : round(_lat, 7),
        'lon'          : round(_lon, 7),
        'x_m'          : round(_x, 2),
        'y_m'          : round(_y, 2),
        'z_m'          : round(_z, 2),
        'dist_from_tx_m': round(_d, 1),
    })

df_coords = pd.DataFrame(rows).sort_values('dist_from_tx_m').reset_index(drop=True)

_out = os.path.join(OUT_DIR, 'receiver_coordinates.csv')
df_coords.to_csv(_out, index=False)

print(f'Receivers  : {len(df_coords)}')
print(f'Dist range : {df_coords["dist_from_tx_m"].min():.0f} – {df_coords["dist_from_tx_m"].max():.0f} m')
print(f'Saved      : {_out}')
print()
print(df_coords.head(10).to_string(index=False))


In [ ]:
# ====================================================================
# CELL 5c — TX/RX POSITION VERIFICATION + 2D OSM MAP (first 50 RX)
# ====================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np, pandas as pd, os, warnings
from shapely.geometry import Point, Polygon, MultiPolygon

print('=' * 65)
print('CELL 5c — POSITION VERIFICATION + 2D MAP')
print('=' * 65)

_safe_v = lambda v: float(v.item()) if hasattr(v, 'item') else float(v)

# ── TX GPS position ───────────────────────────────────────────────────────────
_tx_obj = list(scene.transmitters.values())[0]
_tx_lon_v, _tx_lat_v = local_to_gps(_safe_v(_tx_obj.position[0]),
                                      _safe_v(_tx_obj.position[1]))
print(f'TX  GPS : lon={_tx_lon_v:.6f}  lat={_tx_lat_v:.6f}')

# ── Load first 50 RX from CSV ─────────────────────────────────────────────────
_df_locs = pd.read_csv(RX_CSV)
_first50 = _df_locs.head(50).copy()

# ── Load measured RSSI for first 50 RX ───────────────────────────────────────
_df_meas = pd.read_csv(MEASUREMENT_CSV)
_rssi_col = [c for c in _df_meas.columns if 'measurement' in c.lower()][0]
_name_col = [c for c in _df_meas.columns if 'name' in c.lower()][0]
_rssi_map = dict(zip(_df_meas[_name_col], _df_meas[_rssi_col]))
_first50['rssi_dbm'] = _first50['name'].map(_rssi_map)

# ── Download OSM buildings ─────────────────────────────────────────────────────
print('\nDownloading OSM buildings ...')
_gdf_bld2 = None
try:
    import osmnx as ox
    _ox_ver = tuple(int(x) for x in ox.__version__.split('.')[:2])
    if _ox_ver >= (2, 0):
        _gdf_bld2 = ox.features_from_bbox(
            bbox=(SCENE_WEST, SCENE_SOUTH, SCENE_EAST, SCENE_NORTH),
            tags={'building': True})
    elif _ox_ver >= (1, 3):
        _gdf_bld2 = ox.features_from_bbox(
            bbox=(SCENE_NORTH, SCENE_SOUTH, SCENE_EAST, SCENE_WEST),
            tags={'building': True})
    else:
        _gdf_bld2 = ox.features_from_bbox(
            north=SCENE_NORTH, south=SCENE_SOUTH,
            east=SCENE_EAST, west=SCENE_WEST, tags={'building': True})
    if hasattr(_gdf_bld2, 'crs') and _gdf_bld2.crs and str(_gdf_bld2.crs) != 'EPSG:4326':
        _gdf_bld2 = _gdf_bld2.to_crs('EPSG:4326')
    print(f'  {len(_gdf_bld2):,} buildings downloaded')
except Exception as _e:
    print(f'  OSM download failed: {_e}')

# ── Verify TX not inside building ─────────────────────────────────────────────
if _gdf_bld2 is not None:
    _tx_pt = Point(_tx_lon_v, _tx_lat_v)
    _tx_in = any(
        (isinstance(g, Polygon) and g.contains(_tx_pt)) or
        (isinstance(g, MultiPolygon) and any(p.contains(_tx_pt) for p in g.geoms))
        for g in _gdf_bld2.geometry if g is not None and not g.is_empty
    )
    print(f'\n{"⚠  TX INSIDE building!" if _tx_in else "✓  TX not inside any building"}')

    # Check first 50 RX
    _rx_inside = []
    for _, _r in _first50.iterrows():
        _rpt = Point(float(_r['lon']), float(_r['lat']))
        _inside = any(
            (isinstance(g, Polygon) and g.contains(_rpt)) or
            (isinstance(g, MultiPolygon) and any(p.contains(_rpt) for p in g.geoms))
            for g in _gdf_bld2.geometry if g is not None and not g.is_empty
        )
        if _inside:
            _rx_inside.append(_r['name'])
    if _rx_inside:
        print(f'⚠  {len(_rx_inside)}/50 RX inside buildings: {_rx_inside[:5]}')
    else:
        print(f'✓  All 50 RX outside buildings')

# ── 2D MAP ────────────────────────────────────────────────────────────────────
fig2d, ax2d = plt.subplots(figsize=(11, 10), dpi=150)

# Buildings
if _gdf_bld2 is not None:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        _gdf_bld2.plot(ax=ax2d, facecolor='#d8d0c4', edgecolor='#999999',
                       linewidth=0.15, alpha=0.85, zorder=2)

ax2d.set_xlim(
    min(_first50['lon'].min(), _tx_lon_v) - 0.005,
    max(_first50['lon'].max(), _tx_lon_v) + 0.005)
ax2d.set_ylim(
    min(_first50['lat'].min(), _tx_lat_v) - 0.003,
    max(_first50['lat'].max(), _tx_lat_v) + 0.003)
ax2d.set_aspect('equal')
ax2d.set_facecolor('#eef2f5')
ax2d.set_xlabel('Longitude', fontsize=10)
ax2d.set_ylabel('Latitude',  fontsize=10)
ax2d.tick_params(labelsize=8)
ax2d.grid(True, linestyle='--', linewidth=0.4, alpha=0.5, zorder=1)

# RX dots coloured by measured RSSI
_lons_p = _first50['lon'].values.astype(float)
_lats_p = _first50['lat'].values.astype(float)
_rssi_p = _first50['rssi_dbm'].values.astype(float)
_vmin_p = float(np.nanpercentile(_rssi_p, 2))
_vmax_p = float(np.nanpercentile(_rssi_p, 98))
_sc2d = ax2d.scatter(_lons_p, _lats_p, c=_rssi_p, s=30,
                     cmap='RdYlGn', vmin=_vmin_p, vmax=_vmax_p,
                     alpha=0.85, linewidths=0.4, edgecolors='grey',
                     zorder=5, label='RX (measured RSSI)')

# Number each RX
for _i, (_idx, _r) in enumerate(_first50.iterrows()):
    ax2d.text(float(_r['lon']) + 0.0002, float(_r['lat']) + 0.0001,
              str(_i+1), fontsize=5.5, color='#222222', zorder=7)

_cb2d = fig2d.colorbar(_sc2d, ax=ax2d, fraction=0.025, pad=0.01, shrink=0.7)
_cb2d.set_label('Measured RSSI (dBm)', fontsize=9)
_cb2d.ax.tick_params(labelsize=8)

# TX star
ax2d.plot(_tx_lon_v, _tx_lat_v, marker='*', markersize=18, color='red',
          markeredgecolor='darkred', markeredgewidth=0.8,
          zorder=10, label='TX (transmitter)')
ax2d.annotate('TX', (_tx_lon_v, _tx_lat_v),
              textcoords='offset points', xytext=(8, 5),
              fontsize=9, color='darkred', fontweight='bold', zorder=11)

_n_bld = len(_gdf_bld2) if _gdf_bld2 is not None else 0
ax2d.set_title(
    f'Nottingham 915 MHz — OSM Map: TX + First 50 RX\n'
    f'{_n_bld:,} buildings  |  50 RX coloured by measured RSSI  |  '
    f'RSSI {_vmin_p:.0f}–{_vmax_p:.0f} dBm',
    fontsize=11)
ax2d.legend(loc='upper right', fontsize=9, markerscale=1.5,
            framealpha=0.85, edgecolor='grey')

plt.tight_layout()
_map_out = os.path.join(OUT_DIR, 'tx_rx50_map.png')
plt.savefig(_map_out, dpi=150, bbox_inches='tight')
plt.show()
print(f'\nMap saved: {_map_out}')


In [ ]:
# ====================================================================
# CELL 5d — PER-RECEIVER LOS & BUILDING-INTERIOR CHECK
# ====================================================================
# For each receiver:
#   inside_building : receiver lat/lon is inside an OSM building polygon
#   is_los          : the 2-D line TX→RX does NOT cross any building polygon
#   n_bldgs_crossed : how many building footprints the TX→RX line intersects
# Output saved to results/receiver_los_status.csv
# ====================================================================

import os, math
import numpy as np
import pandas as pd
import osmnx as ox
from shapely.geometry import Point, LineString
from shapely.ops import unary_union
from pyproj import Transformer
import warnings
warnings.filterwarnings('ignore')

os.makedirs('results', exist_ok=True)

# ── 1. Load receiver coordinates (generated by CELL 5b) ─────────────
rx_coord_csv = 'results/receiver_coordinates.csv'
if not os.path.exists(rx_coord_csv):
    raise FileNotFoundError(
        "Run CELL 5b first to generate results/receiver_coordinates.csv")

rx_df = pd.read_csv(rx_coord_csv)
print(f"Loaded {len(rx_df)} receivers from {rx_coord_csv}")

# ── 2. TX position ───────────────────────────────────────────────────
tx_lat = TX_LAT
tx_lon = TX_LON
tx_pt  = Point(tx_lon, tx_lat)   # (lon, lat) — shapely convention

# ── 3. Download OSM buildings ────────────────────────────────────────
# Bounding box that covers all receivers + TX with 200 m margin
all_lats = list(rx_df['lat']) + [tx_lat]
all_lons = list(rx_df['lon']) + [tx_lon]
margin = 0.005   # ~500 m in degrees

bbox = (min(all_lats) - margin, min(all_lons) - margin,
        max(all_lats) + margin, max(all_lons) + margin)

print("Downloading OSM building footprints … ", end='', flush=True)
try:
    gdf_bldg = ox.features_from_bbox(
        bbox=bbox,
        tags={'building': True}
    )
    poly_bldg = [g for g in gdf_bldg.geometry if g.geom_type in ('Polygon','MultiPolygon')]
    print(f"done — {len(poly_bldg)} polygons")
except Exception as e:
    print(f"WARNING: could not fetch OSM buildings: {e}")
    poly_bldg = []

all_buildings_union = unary_union(poly_bldg) if poly_bldg else None

# ── 4. Per-receiver checks ───────────────────────────────────────────
records = []
for _, row in rx_df.iterrows():
    rx_pt  = Point(row['lon'], row['lat'])
    tx_rx_line = LineString([(tx_lon, tx_lat), (row['lon'], row['lat'])])

    inside_building = False
    is_los          = True
    n_bldgs_crossed = 0

    if all_buildings_union is not None:
        # Inside-building check
        inside_building = bool(rx_pt.within(all_buildings_union))

        # LOS check: count how many individual building polys the line crosses
        for poly in poly_bldg:
            try:
                if tx_rx_line.crosses(poly) or tx_rx_line.within(poly):
                    n_bldgs_crossed += 1
            except Exception:
                pass
        is_los = (n_bldgs_crossed == 0)

    records.append({
        'id':              int(row['id']),
        'lat':             row['lat'],
        'lon':             row['lon'],
        'dist_m':          row['dist_from_tx_m'],
        'inside_building': inside_building,
        'is_los':          is_los,
        'n_bldgs_crossed': n_bldgs_crossed,
    })

out_df = pd.DataFrame(records).sort_values('dist_m').reset_index(drop=True)

# ── 5. Summary ───────────────────────────────────────────────────────
n_inside = out_df['inside_building'].sum()
n_los    = out_df['is_los'].sum()
n_nlos   = (~out_df['is_los']).sum()
print(f"\n{'─'*50}")
print(f"Total receivers    : {len(out_df)}")
print(f"Inside building    : {n_inside}  ({100*n_inside/len(out_df):.1f}%)")
print(f"Clear LOS          : {n_los}   ({100*n_los/len(out_df):.1f}%)")
print(f"NLOS (obstructed)  : {n_nlos}  ({100*n_nlos/len(out_df):.1f}%)")
print(f"{'─'*50}")

# ── 6. Save CSV ──────────────────────────────────────────────────────
out_path = 'results/receiver_los_status.csv'
out_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(out_df[['id','dist_m','inside_building','is_los','n_bldgs_crossed']].head(20).to_string(index=False))


## Cell A — paths.a Normalization Diagnostic

Compares `sum(|paths.a|²)` to theoretical FSPL at 3 distances. Reveals if Sionna 2.0 PathSolver has different amplitude normalization than expected. Run after Cell 6 (receivers placed), before DIAG.

In [ ]:
# ====================================================================
# CELL A — paths.a NORMALIZATION DIAGNOSTIC
# ====================================================================
# Runs a pure LOS path solve at 3 known distances, prints paths.a.shape
# and compares sum(|a|²) to FSPL — reveals any normalization offset.
# Run after Cell 4 (TX placed) and Cell 6 (RX list loaded).
# ====================================================================
import numpy as np, math, time
print("=" * 70)
print("CELL A — paths.a NORMALIZATION CHECK (LOS comparison to FSPL)")
print("=" * 70)

C = 3e8
_lam = C / FREQUENCY_HZ  # wavelength

def _fspl_linear(d):
    """Free-space path gain (linear) = (λ/4πr)²"""
    return (_lam / (4 * math.pi * d)) ** 2

def _fspl_db(d):
    return -10 * math.log10(_fspl_linear(d))

tx_obj = list(scene.transmitters.values())[0]
_tx_lx = _safe(tx_obj.position[0])
_tx_ly = _safe(tx_obj.position[1])
_tx_lz = _safe(tx_obj.position[2])

print(f"\nWavelength     : {_lam:.4f} m")
print(f"TX position    : ({_tx_lx:.1f}, {_tx_ly:.1f}, {_tx_lz:.1f}) m")
print(f"RX AGL         : {RX_AGL_M} m")
print()
print(f"  {'Dist':>6}  {'FSPL(dB)':>9}  {'a.shape(raw)':>22}  {'sum|a|²(dB)':>12}  {'vs FSPL':>9}  {'paths':>6}  {'time(s)':>7}")
print(f"  {'-'*90}")

_test_dists = [50, 200, 500, 1000, 2000]

for _test_d in _test_dists:
    _rx_lx = _tx_lx + _test_d  # due East
    _rx_lz = RX_AGL_M
    _rx_test = Receiver(name='_debug_rx', position=[_rx_lx, _tx_ly, _rx_lz])
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    scene.add(_rx_test)

    _t0 = time.time()
    try:
        _paths = PathSolver()(scene,
            max_depth=MAX_DEPTH,
            los=True,
            specular_reflection=True,
            diffraction=True,
            edge_diffraction=True,
            diffuse_reflection=True,
            samples_per_src=2_000_000)
        _dt = time.time() - _t0

        # Combine real/imag tuple → complex array
        _a_raw = _paths.a
        if isinstance(_a_raw, tuple):
            _raw_shape = ('tuple', tuple(_a_raw[0].shape))
            _a_np = ((_a_raw[0].numpy() if hasattr(_a_raw[0], 'numpy') else np.array(_a_raw[0])) +
                     1j*(_a_raw[1].numpy() if hasattr(_a_raw[1], 'numpy') else np.array(_a_raw[1])))
        else:
            _raw_shape = tuple(_a_raw.shape)
            _a_np = _a_raw.numpy() if hasattr(_a_raw, 'numpy') else np.array(_a_raw)

        # Squeeze
        _a_sq = np.squeeze(_a_np)

        # Print shapes for debug
        _sum_pwr = float(np.sum(np.abs(_a_sq) ** 2))
        _n_valid = int(np.sum(np.abs(_a_sq) ** 2 > 1e-30))
        if _sum_pwr > 1e-30:
            _sum_db = 10 * math.log10(_sum_pwr)
        else:
            _sum_db = float('nan')

        _fspl = _fspl_db(_test_d)
        _vs_fspl = _sum_db - (-_fspl)  # sum_db is negative, FSPL is positive loss
        # correct comparison: sum|a|² (dB) should ≈ -FSPL (negative)
        _vs_fspl2 = _sum_db - (-_fspl)

        print(f"  {_test_d:>6}m  {_fspl:>9.1f}  {str(_raw_shape):>22}  {_sum_db:>12.2f}  {(_sum_db + _fspl):>+9.1f}  {_n_valid:>6}  {_dt:>7.1f}")

    except Exception as _e:
        _dt = time.time() - _t0
        print(f"  {_test_d:>6}m  ERROR: {_e}  ({_dt:.1f}s)")

print()
print("  Expected: sum|a|²(dB) ≈ -(FSPL dB) for open LOS.")
print("  'vs FSPL' = sum|a|²(dB) + FSPL(dB)  (should be ~0 to +10 dB for urban overhead)")
print()
print("  paths.a raw shape legend:")
print("  Sionna 2.0 typical: [num_rx, num_tx, num_rx_ant, num_tx_ant, num_paths]")
print("  or: [batch, num_rx, ...] — check documentation for your version")
print()

# Restore receivers
for _n in list(scene.receivers.keys()): scene.remove(_n)
_rxlist_debug = list(receivers) if 'receivers' in dir() else []
for _rx in _rxlist_debug: scene.add(_rx)
print(f"  Receivers restored: {len(_rxlist_debug)}")
print()
print("  ── Top 5 paths for last test distance ──────────────────────────────")
try:
    _rx_test2 = Receiver(name='_debug_rx2', position=[_tx_lx + 200, _tx_ly, RX_AGL_M])
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    scene.add(_rx_test2)
    _paths2 = PathSolver()(scene,
        max_depth=MAX_DEPTH, los=True, specular_reflection=True,
        diffraction=True, edge_diffraction=True, diffuse_reflection=True,
        samples_per_src=2_000_000)
    _a2_raw = _paths2.a
    if isinstance(_a2_raw, tuple):
        _a2 = np.squeeze(
            (_a2_raw[0].numpy() if hasattr(_a2_raw[0], 'numpy') else np.array(_a2_raw[0])) +
            1j*(_a2_raw[1].numpy() if hasattr(_a2_raw[1], 'numpy') else np.array(_a2_raw[1])))
        print(f"  paths.a: tuple(real,imag), real shape={_a2_raw[0].shape}")
    else:
        _a2 = np.squeeze(_a2_raw.numpy() if hasattr(_a2_raw, 'numpy') else np.array(_a2_raw))
    print(f"  paths.a after squeeze shape: {_a2.shape}  dtype: {_a2.dtype}")
    _pwr2 = np.abs(_a2.flatten()) ** 2
    _ord2 = np.argsort(_pwr2)[::-1]
    for _ri in range(min(5, len(_ord2))):
        _pi = _ord2[_ri]
        _pw = _pwr2[_pi]
        if _pw > 1e-40:
            print(f"    rank {_ri+1}: |a|²={_pw:.4e}  ({10*math.log10(_pw):.1f} dB)")
    print(f"  Expected LOS |a|² at 200m = {_fspl_linear(200):.4e}  ({-_fspl_db(200):.1f} dB)")
    # Friis check
    _rssi_check = TX_CONDUCTED_DBM + 10*math.log10(max(_pwr2[_ord2[0]], 1e-40)) + RX_EXTRA_GAIN_DB
    print(f"  RSSI from strongest path: {_rssi_check:.1f} dBm  (expected ~{TX_CONDUCTED_DBM - _fspl_db(200) + RX_EXTRA_GAIN_DB:.1f} dBm for FSPL)")
    # Tau
    if hasattr(_paths2, 'tau'):
        _tau2 = np.squeeze(np.array(_paths2.tau))
        print(f"  paths.tau shape: {_tau2.shape}")
        _tau_flat = _tau2.flatten()
        _valid_tau = _tau_flat[_tau_flat > 0]
        if len(_valid_tau):
            _los_tau = 200 / C
            print(f"  Expected LOS tau: {_los_tau:.6f} s  ({200}m/c)")
            print(f"  Min tau found: {_valid_tau.min():.6f} s  ({_valid_tau.min()*C:.1f}m)")
except Exception as _e2:
    print(f"  ERROR: {_e2}")
    import traceback; traceback.print_exc()
finally:
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    for _rx in _rxlist_debug: scene.add(_rx)
    print(f"  Receivers restored: {len(_rxlist_debug)}")


## Cell DIAG — Step-by-Step Bias Diagnostic

Run **before the path solver** to verify TX/RX positions, antenna heights, and scene geometry.
Tests 10 → 100 receivers to catch systematic bias early. Uses Sionna 2.0 `compute_paths()` API.

In [ ]:
from sionna.rt import PathSolver
# ====================================================================
# CELL DIAG — Step-by-Step Bias Diagnostic (10 → 100 receivers)
# ====================================================================
# Tests RSSI formula, RX heights, TX position, and geometry systematically.
# Run BEFORE CELL 9b to isolate the source of high RMSE.
# ====================================================================
import numpy as np, pandas as pd, math, os, time
from pyproj import Transformer as _Tr

print("=" * 70)
print("BIAS DIAGNOSTIC — Step-by-step RMSE decomposition")
print("=" * 70)

# ── Load measurements ─────────────────────────────────────────────────────────
_df_meas = pd.read_csv(MEASUREMENT_CSV)
print(f"\nMeasurements loaded: {len(_df_meas)} rows")
print(f"  RSSI range   : {_df_meas['local_measurement_dBm'].min():.1f} → {_df_meas['local_measurement_dBm'].max():.1f} dBm")
print(f"  PL range     : {_df_meas['path_loss_dB'].min():.1f} → {_df_meas['path_loss_dB'].max():.1f} dB")

# ── STEP 1: Formula check — FSPL vs measured at known distances ────────────────
print("\n" + "─" * 60)
print("STEP 1 — Free-Space Path Loss formula validation")
print("─" * 60)
C = 3e8
_f = FREQUENCY_HZ
_fspl_fn = lambda d: 20*np.log10(4*np.pi*d*_f/C)

_tx_lon, _tx_lat = TX_LON, TX_LAT
_gps2utm = Transformer.from_crs("EPSG:4326", f"EPSG:{UTM_EPSG}", always_xy=True)
_tx_x, _tx_y = _gps2utm.transform(_tx_lon, _tx_lat)

_rx_x = np.array([_gps2utm.transform(r['lon'], r['lat'])[0] for _, r in _df_meas.iterrows()])
_rx_y = np.array([_gps2utm.transform(r['lon'], r['lat'])[1] for _, r in _df_meas.iterrows()])
_dist = np.sqrt((_rx_x - _tx_x)**2 + (_rx_y - _tx_y)**2)
_df_meas = _df_meas.copy()
_df_meas['dist_m'] = _dist

# Near receivers (50-300m) — most likely LOS → compare against FSPL
_near = _df_meas[_df_meas['dist_m'].between(50, 300)].copy()
_near['fspl_db']    = _near['dist_m'].apply(_fspl_fn)
_near['measured_pl'] = _near['path_loss_dB']
_near['vs_fspl']    = _near['measured_pl'] - _near['fspl_db']
print(f"  Near receivers (50-300m): {len(_near)}")
print(f"  TX_CONDUCTED={TX_CONDUCTED_DBM:.1f} dBm  RX_EXTRA={RX_EXTRA_GAIN_DB:.1f} dB  SITE_CORR={SITE_CORRECTION_DB:.1f} dB")
print(f"  RSSI formula: RSSI = TX_CONDUCTED - PL + RX_EXTRA + SITE_CORR")
print(f"              = {TX_CONDUCTED_DBM:.1f} - PL + {RX_EXTRA_GAIN_DB:.1f} + {SITE_CORRECTION_DB:.1f}")
print(f"  → PL = {TX_CONDUCTED_DBM + RX_EXTRA_GAIN_DB + SITE_CORRECTION_DB:.1f} - RSSI  (= rssi_from_path_gain inverse)")
print()
print(f"  {'Name':<12} {'Dist(m)':>8} {'RSSI(dBm)':>10} {'PL_meas(dB)':>12} {'FSPL(dB)':>9} {'PL-FSPL(dB)':>12}")
for _, r in _near.head(10).iterrows():
    print(f"  {str(r['name']):<12} {r['dist_m']:>8.0f} {r['local_measurement_dBm']:>10.1f} "
          f"{r['measured_pl']:>12.1f} {r['fspl_db']:>9.1f} {r['vs_fspl']:>12.1f}")
_near_overhead = _near['vs_fspl'].mean()
print(f"\n  Mean PL-FSPL (near): {_near_overhead:+.1f} dB  "
      f"(expected +5 to +15 dB for urban LOS overhead)")
if _near_overhead < 0:
    print(f"  ⚠ Negative overhead → TX_CONDUCTED+RX_EXTRA+SITE_CORR over-estimated")
elif _near_overhead > 25:
    print(f"  ⚠ Very high overhead → TX power under-estimated or RX underground")
else:
    print(f"  ✓ Urban overhead looks physically reasonable")

# ── STEP 2: RX height check ────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("STEP 2 — Receiver height sanity check")
print("─" * 60)
_rxlist = list(scene.receivers.values())
if not _rxlist and os.path.exists(RX_CSV):
    _df_rx2 = pd.read_csv(RX_CSV)
    for _, _row2 in _df_rx2.iterrows():
        _x2  = float(_row2['x_m'])
        _y2  = float(_row2['y_m'])
        _lz2 = float(_row2['z_m'])
        _rx2 = Receiver(name=str(_row2['name']), position=[_x2, _y2, _lz2])
        scene.add(_rx2)
    _rxlist = list(scene.receivers.values())
    print(f"  Loaded {len(_rxlist)} receivers from {RX_CSV} (terrain-corrected z)")
_heights = [_safe(rx.position[2]) for rx in _rxlist[:20]]
print(f"  First 20 RX heights (local Z, m):")
for rx, h in zip(_rxlist[:20], _heights):
    flag = " ⚠ UNDERGROUND" if h < -5 else (" ⚠ TOO HIGH (check terrain)" if h > 300 else "")
    print(f"    {rx.name:<12}  z={h:+.2f}m{flag}")
_neg = sum(1 for h in [_safe(rx.position[2]) for rx in _rxlist] if h < 0)
print(f"\n  Total RX underground (z<0): {_neg} / {len(_rxlist)}")

# ── STEP 3: TX position check ─────────────────────────────────────────────────
print("\n" + "─" * 60)
print("STEP 3 — TX position check")
print("─" * 60)
if 'TX_AGL_M' not in dir(): TX_AGL_M = 17.0
_tx = list(scene.transmitters.values())[0]
_tx_lx, _tx_ly, _tx_lz = _safe(_tx.position[0]), _safe(_tx.position[1]), _safe(_tx.position[2])
_tx_glon, _tx_glat = local_to_gps(_tx_lx, _tx_ly)
print(f"  TX local  : ({_tx_lx:.1f}, {_tx_ly:.1f}, {_tx_lz:.1f}) m")
print(f"  TX GPS    : lat={_tx_glat:.6f}  lon={_tx_glon:.6f}")
print(f"  Expected  : lat={TX_LAT:.6f}  lon={TX_LON:.6f}  h={TX_AGL_M:.1f}m")
_lat_err = abs(_tx_glat - TX_LAT) * 111000
_lon_err = abs(_tx_glon - TX_LON) * 111000 * math.cos(math.radians(TX_LAT))
print(f"  Position error: {_lat_err:.1f}m N-S  {_lon_err:.1f}m E-W  Z={_tx_lz:.1f}m (terrain+AGL)")
print(f"  Terrain at TX  : {_tx_lz - TX_AGL_M:.1f}m  AGL={TX_AGL_M:.1f}m  Total={_tx_lz:.1f}m ✓")
print(f"  TX height: AGL={TX_AGL_M:.1f}m  terrain_z={_tx_lz-TX_AGL_M:.1f}m  total={_tx_lz:.1f}m")
if _lat_err > 50 or _lon_err > 50:
    print("  ⚠ TX position error > 50m — check GPS→UTM→local conversion")
else:
    print("  ✓ TX position OK")

# ── STEP 4: Quick path solver on 50 receivers across bands ─────────────────────
print("\n" + "─" * 60)
print("STEP 4 — Path solver: 50 receivers across distance bands")
print("─" * 60)

# ── Scatter correction helpers (mirrors Cell 9b) ──────────────────────────────
_DIAG_SCAT_KEEP = 1.0  # Sionna 2.0 — deterministic, no scat_keep_prob correction

def _diag_extract_types(paths, shape_2d):
    """Return per-path type array (0=LOS,1=refl,2=diff,3=scat) or None."""
    for attr in ('types', 'type', 'interactions', 'interaction_types'):
        t = getattr(paths, attr, None)
        if t is None: continue
        try:
            t_np = t.numpy() if hasattr(t, 'numpy') else np.array(t)
            t_np = np.squeeze(t_np)
            while t_np.ndim > 2: t_np = t_np[..., 0]
            if t_np.ndim == 1: t_np = t_np[np.newaxis, :]
            if t_np.shape == shape_2d: return t_np.astype(np.int8)
        except Exception: pass
    return None

def _diag_summary(a_row, types_row=None):
    """Path gain → incoherent power linear. Sionna 2.0: no scat_keep_prob correction."""
    pwr = np.abs(a_row) ** 2
    valid = pwr > 1e-30
    if not np.any(valid): return None, 0
    pv = pwr[valid].copy(); av = a_row[valid].copy()
    corr = 1.0 / _DIAG_SCAT_KEEP
    if types_row is not None:
        tv = types_row[valid]; scat_mask = (tv == 3)
        if np.any(scat_mask): pv[scat_mask] *= corr
    else:
        _mx = np.max(pv)
        _heur = pv < (_mx * 1e-3)
        if np.any(_heur): pv[_heur] *= corr
        elif _DIAG_SCAT_KEEP < 1.0: pv *= corr
    return float(np.sum(pv)), int(np.sum(valid))

# ── Distance bands ────────────────────────────────────────────────────────────
_gps2utm3 = Transformer.from_crs("EPSG:4326", f"EPSG:{UTM_EPSG}", always_xy=True)
_df_meas2 = _df_meas.copy()
_xy = _df_meas2.apply(lambda r: _gps2utm3.transform(float(r['lon']), float(r['lat'])), axis=1)
_df_meas2['_lx'] = [xy[0] - utm_center_x for xy in _xy]
_df_meas2['_ly'] = [xy[1] - utm_center_y for xy in _xy]
_df_meas2['_dtx'] = np.sqrt((_df_meas2['_lx'] - _tx_lx)**2 + (_df_meas2['_ly'] - _tx_ly)**2)
_df_meas2 = _df_meas2.sort_values('_dtx').reset_index(drop=True)
# Filter out near-TX outliers (< 50m) — GPS errors / mast co-location artefacts
_df_meas2 = _df_meas2[_df_meas2['_dtx'] >= 100].copy()
_bands_sel = [
    _df_meas2[_df_meas2['_dtx'].between( 100,  300)].head(10),
    _df_meas2[_df_meas2['_dtx'].between( 300,  700)].head(10),
    _df_meas2[_df_meas2['_dtx'].between( 700, 1200)].head(10),
    _df_meas2[_df_meas2['_dtx'].between(1200, 2000)].head(10),
    _df_meas2[_df_meas2['_dtx'] > 2000].head(10),
]
_df_test = pd.concat(_bands_sel).drop_duplicates(subset='name').reset_index(drop=True)
print(f"  Testing {len(_df_test)} receivers across 5 distance bands")
print(f"  Formula: rssi_from_path_gain() — TX_CONDUCTED={TX_CONDUCTED_DBM:.1f} RX_EXTRA={RX_EXTRA_GAIN_DB:.1f} SITE_CORR={SITE_CORRECTION_DB:.1f}")
print("  Scatter correction: disabled (Sionna 2.0 deterministic solver)")
print(f"  Samples: 10M per RX  |  max_depth: {MAX_DEPTH}")

_results50 = []
for _, _mrow in _df_test.iterrows():
    _lx = float(_mrow['_lx']); _ly = float(_mrow['_ly']); _d = float(_mrow['_dtx'])
    _rssi_meas = float(_mrow['local_measurement_dBm'])
    _pl_meas   = float(_mrow['path_loss_dB'])
    _rx_name   = str(_mrow['name'])
    # Use position already set by Cell 6c/7 in scene.receivers — these are
    # terrain-corrected and match the positions used in Cell 9b.
    # Recomputing from CSV or DEM risks using a different elevation reference.
    _scene_pos = {rx.name: rx for rx in _rxlist}
    if _rx_name in _scene_pos:
        _rx_z = float(_safe(_scene_pos[_rx_name].position[2]))
        _gz4  = _rx_z - RX_AGL_M
    else:
        # Fallback: DEM lookup (scene-local = ASL - scene_centre_DTM)
        try:
            _ux4, _uy4 = gps_to_utm.transform(float(_mrow["lon"]), float(_mrow["lat"]))
            _gz4 = get_dem_elevation(_ux4 - utm_center_x, _uy4 - utm_center_y)
        except Exception: _gz4 = 0.0
        _rx_z = _gz4 + RX_AGL_M
    print(f'    [{_rx_name}] ground_z={_gz4:.2f}m  rx_z={_rx_z:.2f}m')
    _rx = Receiver(name=_rx_name, position=[_lx, _ly, _rx_z])
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    scene.add(_rx)
    try:
        _t0 = time.time()
        _paths = PathSolver()(scene, 
            max_depth=MAX_DEPTH, los=True, specular_reflection=True,
            diffraction=True, edge_diffraction=True, diffuse_reflection=True,
 samples_per_src=10_000_000)
        _dt = time.time() - _t0
        _a_raw = _paths.a
        if isinstance(_a_raw, tuple):
            _a = (_a_raw[0].numpy() if hasattr(_a_raw[0], 'numpy') else np.array(_a_raw[0])) + \
                 1j*(_a_raw[1].numpy() if hasattr(_a_raw[1], 'numpy') else np.array(_a_raw[1]))
            print(f'      paths.a: tuple(real,imag), each shape {_a_raw[0].shape}')
        else:
            _a = _a_raw.numpy() if hasattr(_a_raw, 'numpy') else np.array(_a_raw)
            print(f'      paths.a raw shape: {_a.shape}  dtype: {_a.dtype}')
        _a = np.squeeze(_a)
        print(f'      paths.a after squeeze: {_a.shape}  dtype: {_a.dtype}')
        # Normalise to 2-D (n_rx, n_paths) — squeeze can produce 0-D or 1-D scalars
        if _a.ndim == 0: _a = _a.reshape(1, 1)
        elif _a.ndim == 1: _a = _a[np.newaxis, :]
        elif _a.ndim > 2: _a = _a.reshape(1, -1)  # flatten extra dims into paths
        _t_all = _diag_extract_types(_paths, _a.shape)
        _t_row = _t_all[0] if (_t_all is not None and _t_all.ndim > 1) else _t_all
        _pg_sum, _n_paths = _diag_summary(_a[0], _t_row)
        if _pg_sum and _pg_sum > 0:
            _rssi_sim = rssi_from_path_gain(_pg_sum)
            _pl_sim   = -10*math.log10(_pg_sum)
        else:
            _rssi_sim = _pl_sim = float('nan')
            _n_paths  = 0
        _fspl_d = 20*math.log10(4*math.pi*_d*FREQUENCY_HZ/C) if _d > 0 else 0
        _results50.append({'name': _rx_name, 'dist_m': _d,
                           'rssi_sim': _rssi_sim, 'rssi_meas': _rssi_meas,
                           'pl_sim': _pl_sim, 'pl_meas': _pl_meas,
                           'fspl': _fspl_d, 'n_paths': _n_paths, 'dt': _dt})
        _err = _rssi_sim - _rssi_meas if not math.isnan(_rssi_sim) else float('nan')
        _ovr = _pl_sim - _fspl_d if not math.isnan(_pl_sim) else float('nan')
        print(f"  {_rx_name:<12} d={_d:5.0f}m  paths={_n_paths:3d}  "
              f"RSSI sim={_rssi_sim:6.1f} meas={_rssi_meas:6.1f}  "
              f"err={_err:+5.1f}  PL-FSPL={_ovr:+5.1f} dB  ({_dt:.1f}s)")
    except Exception as _e:
        print(f"  {_rx_name:<12} ERROR: {_e}")

# Re-add all receivers
for _n in list(scene.receivers.keys()): scene.remove(_n)
for _rx in _rxlist: scene.add(_rx)

_r10_raw = pd.DataFrame(_results50).dropna(subset=['rssi_sim', 'rssi_meas'])
_RSSI_MAX_VALID = -5.0; _DIST_MIN_VALID = 50.0
_r10_clamped = _r10_raw[(_r10_raw['rssi_sim'] <= _RSSI_MAX_VALID) &
                         (_r10_raw['dist_m']   >= _DIST_MIN_VALID)].copy()
_n_removed = len(_r10_raw) - len(_r10_clamped)
if _n_removed:
    print(f"  Clamped {_n_removed} anomalous RX (sim RSSI>{_RSSI_MAX_VALID}dBm or dist<{_DIST_MIN_VALID}m):")
    for _, _rr in _r10_raw[~_r10_raw.index.isin(_r10_clamped.index)].iterrows():
        print(f"    {_rr['name']:<14} d={_rr['dist_m']:4.0f}m  sim={_rr['rssi_sim']:+.1f}dBm  meas={_rr['rssi_meas']:+.1f}dBm  → EXCLUDED")

_r50 = _r10_clamped
if len(_r50):
    _bias = (_r50['rssi_sim'] - _r50['rssi_meas']).mean()
    _rmse = math.sqrt(((_r50['rssi_sim'] - _r50['rssi_meas'])**2).mean())
    print(f"\n  Valid RX summary ({len(_r50)} receivers, dist≥{_DIST_MIN_VALID:.0f}m):")
    print(f"  bias={_bias:+.1f} dB  RMSE={_rmse:.1f} dB")
    print(f"  PL vs FSPL : mean={(_r50['pl_sim']-_r50['fspl']).mean():+.1f} dB  (urban expect +5 to +20 dB)")
    print(f"\n  Distance-band breakdown:")
    print(f"  {'Band':<12} {'N':>4} {'Bias(dB)':>10} {'RMSE(dB)':>10} {'Mean paths':>11}")
    print(f"  {'-'*52}")
    for _bname, _bmin, _bmax in [('<300m',0,300),('300-700m',300,700),
                                   ('700-1200m',700,1200),('1.2-2km',1200,2000),('>2km',2000,9999)]:
        _rb = _r50[(_r50['dist_m']>=_bmin) & (_r50['dist_m']<_bmax)]
        if not len(_rb): continue
        _berr = _rb['rssi_sim'] - _rb['rssi_meas']
        print(f"  {_bname:<12} {len(_rb):>4} {_berr.mean():>+10.1f} "
              f"{float(np.sqrt((_berr**2).mean())):>10.1f} {_rb['n_paths'].mean():>11.0f}")
    if abs(_bias) <= 5 and _rmse <= 10:
        print(f"\n  ✓ Formula and geometry look correct — proceed to Cell 9b")
    elif abs(_bias) > 10:
        print(f"\n  ⚠ Large bias={_bias:+.1f} dB — check TX_CONDUCTED / RX_EXTRA / SITE_CORRECTION")
    else:
        print(f"\n  ⚠ Large scatter RMSE={_rmse:.1f} dB — likely scene geometry gaps (run CELL 3c)")

# ── STEP 5: Distance-band RMSE using ALL scene receivers ─────────────────────
print("\n" + "─" * 60)
print("STEP 5 — Distance-band RSSI vs FSPL reference (all scene receivers)")
print("─" * 60)
try:
    _scene_rx = {nm: rx for nm, rx in scene.receivers.items()}
    _tx_sx = _safe(tx.position[0]); _tx_sy = _safe(tx.position[1])
    _df_band = pd.read_csv(MEASUREMENT_CSV).dropna(subset=['local_measurement_dBm'])
    _rows = []
    for _, _row in _df_band.iterrows():
        _nm = str(_row['name'])
        if _nm not in _scene_rx: continue
        _rx = _scene_rx[_nm]
        _d3 = math.sqrt((_safe(_rx.position[0])-_tx_sx)**2 + (_safe(_rx.position[1])-_tx_sy)**2)
        _rows.append({'name': _nm, 'dist': _d3, 'rssi_meas': float(_row['local_measurement_dBm'])})
    _df_band2 = pd.DataFrame(_rows)
    print(f"  Matched {len(_df_band2)} receivers  |  dist range: {_df_band2['dist'].min():.0f}–{_df_band2['dist'].max():.0f}m")
    print(f"  Reference: rssi_from_path_gain(FSPL_linear)  [free-space upper bound]\n")
    print(f"  {'Band':<12} {'N':>5}  {'Mean dist':>10}  {'Bias(dB)':>10}  {'RMSE(dB)':>10}")
    print(f"  {'-'*56}")
    for (d0, d1), lbl in zip([(0,100),(100,500),(500,1000),(1000,2000),(2000,99999)],
                               ['0–100m','100–500m','500m–1km','1–2km','>2km']):
        _sub = _df_band2[(_df_band2['dist'] >= d0) & (_df_band2['dist'] < d1)]
        if len(_sub) < 2:
            print(f"  {lbl:<12} {len(_sub):>5}  {'—':>10}  {'—':>10}  {'—':>10}"); continue
        _fspl_lin = (3e8 / (4*np.pi*_sub['dist']*FREQUENCY_HZ))**2
        _rssi_fs  = rssi_from_path_gain(_fspl_lin)   # free-space RSSI upper bound
        _bias_b   = (_rssi_fs - _sub['rssi_meas']).mean()
        _rmse_b   = math.sqrt(((_rssi_fs - _sub['rssi_meas'])**2).mean())
        print(f"  {lbl:<12} {len(_sub):>5}  {_sub['dist'].mean():>9.0f}m  {_bias_b:>+9.1f}  {_rmse_b:>9.1f}")
    print(f"\n  NOTE: STEP 5 uses free-space as reference, not ray tracing.")
    print(f"  Run CELL 9b + 9d for full sim-vs-meas RMSE.")
except Exception as _e5:
    print(f"  STEP 5 error: {_e5}")
    import traceback; traceback.print_exc()


## Cell 7 — Path Solver (900 MHz, Sionna 2.0, Batched)

Full path solver with per-ray extraction, adaptive samples, and CSV output.
Uses Sionna 2.0 API — no `scat_keep_prob` parameter.

In [ ]:
# ====================================================================
# CELL 7 — PATH SOLVER WITH PER-RAY EXTRACTION  [900 MHz / Sionna 2.0]
# ====================================================================
# Processes all receivers in batches of BATCH_SIZE using compute_paths().
# Sionna 2.0 API: no scat_keep_prob parameter.
# paths.a is a complex tensor — power per path = |paths.a|²
# ====================================================================
import gc, time, os, math
import numpy as np, pandas as pd
from datetime import datetime

print('=' * 70)
print('CELL 7 — PATH SOLVER  [900 MHz / Sionna 2.0]')
print('=' * 70)

_safe_ps = lambda v: float(v.item()) if hasattr(v, 'item') else float(v)

# ── Configuration ─────────────────────────────────────────────────────────────
SAVE_PER_RAY    = True
MAX_RAYS_PER_RX = 300
MAX_SAMPLES_PS  = 20_000_000   # hard cap for OOM safety

# Sionna 2.0 deterministic solver — no scat_keep_prob
PS_CONFIG_BASE = dict(
    max_depth        = MAX_DEPTH,
    los              = True,
    specular_reflection = True,
    diffraction      = True,
    diffuse_reflection  = True,
    edge_diffraction = True,
    # no scat_keep_prob in Sionna 2.0
)

_tx_ps = list(scene.transmitters.values())[0]
tx_pos = np.array([_safe_ps(_tx_ps.position[0]),
                   _safe_ps(_tx_ps.position[1]),
                   _safe_ps(_tx_ps.position[2])])
_C_local = 3e8

print(f'  Sionna 2.0 deterministic solver — no scat_keep_prob')
print(f'  TX conducted    : {TX_CONDUCTED_DBM:.1f} dBm  |  RX extra: {RX_EXTRA_GAIN_DB:.1f} dB  |  Site corr: {SITE_CORRECTION_DB:.1f} dB')
print(f'  TX position     : ({tx_pos[0]:.1f}, {tx_pos[1]:.1f}, {tx_pos[2]:.1f}) m')
print(f'  Batch size      : {BATCH_SIZE} receivers  |  Base samples: {NUM_SAMPLES_PS:,}')
for k, v in PS_CONFIG_BASE.items():
    print(f'  {k:15s}: {v}')

def adaptive_samples(dist_m):
    if dist_m > 9000:   return min(NUM_SAMPLES_PS * 2, MAX_SAMPLES_PS)
    elif dist_m > 5000: return min(int(NUM_SAMPLES_PS * 1.5), MAX_SAMPLES_PS)
    else:               return NUM_SAMPLES_PS

def extract_amplitudes(paths):
    """Extract complex CIR amplitudes from Sionna 2.0 Paths object.

    In this Sionna 2.0 build paths.a returns a (real, imag) tuple of
    float32 tensors.  Combine them before any further processing.
    Shape after combining: [num_rx, num_paths] complex64.
    """
    def _to_np(t):
        return t.numpy() if hasattr(t, 'numpy') else np.array(t)

    a = getattr(paths, 'a', None)
    if a is not None:
        try:
            if isinstance(a, tuple):
                a_np = _to_np(a[0]) + 1j * _to_np(a[1])
            else:
                a_np = _to_np(a)
            a_np = np.squeeze(a_np)
            if a_np.ndim == 1:
                a_np = a_np[np.newaxis, :]
            elif a_np.ndim > 2:
                a_np = a_np.reshape(a_np.shape[0], -1)
            return a_np.astype(complex)
        except Exception:
            pass
    # Fallback: paths.cir()
    try:
        a_t, _ = paths.cir()
        if isinstance(a_t, tuple):
            a_np = _to_np(a_t[0]) + 1j * _to_np(a_t[1])
        else:
            a_np = _to_np(a_t)
        a_np = np.squeeze(a_np)
        if a_np.ndim == 1:
            a_np = a_np[np.newaxis, :]
        elif a_np.ndim > 2:
            a_np = a_np.reshape(a_np.shape[0], -1)
        return a_np.astype(complex)
    except Exception:
        return np.zeros((1, 1), dtype=complex)

def extract_tau(paths, num_rx, num_paths):
    tau = getattr(paths, 'tau', None)
    if tau is None:
        return np.full((num_rx, num_paths), np.nan, np.float32)
    try:
        t = tau.numpy() if hasattr(tau, 'numpy') else np.array(tau)
        t = np.squeeze(t)
        while t.ndim > 2: t = t[..., 0]
        if t.ndim == 1: t = t[np.newaxis, :]
        return t
    except Exception:
        return np.full((num_rx, num_paths), np.nan, np.float32)

def summary_metrics(a_row):
    """Compute best/incoherent/coherent path loss from complex amplitude row.
    Sionna 2.0 — no scat_keep_prob correction needed.
    """
    pwr   = np.abs(a_row) ** 2
    valid = pwr > 1e-30
    if not np.any(valid):
        return np.nan, np.nan, np.nan, 0
    pv = pwr[valid]
    av = a_row[valid]
    best_pl       = -10 * np.log10(np.max(pv))
    incoherent_pl = -10 * np.log10(np.sum(pv))
    coh_pwr       = np.abs(np.sum(av)) ** 2
    coherent_pl   = -10 * np.log10(coh_pwr) if coh_pwr > 1e-30 else np.nan
    return best_pl, incoherent_pl, coherent_pl, int(np.sum(valid))

def ray_type_heuristic(path_len, los_dist, pwr, max_pwr):
    if los_dist > 0 and abs(path_len - los_dist) / los_dist < 0.01: return 'LOS'
    ratio = pwr / max_pwr if max_pwr > 0 else 0
    excess = path_len - los_dist
    if excess < 50  and ratio > 0.01:  return 'REFLECTION'
    if excess >= 50 and ratio > 0.001: return 'MULTI_REFLECTION'
    if ratio < 0.01:                   return 'DIFFRACTION'
    if ratio < 0.001:                  return 'SCATTERING'
    return 'UNKNOWN'

def run_batch(batch, cfg):
    for nm in list(scene.receivers.keys()): scene.remove(nm)
    for rx in batch: scene.add(rx)
    try:
        paths = PathSolver()(scene, **cfg)
    except Exception as _oom:
        if any(k in str(_oom).lower() for k in ['oom', 'resource exhausted', 'memory']):
            paths = scene.compute_paths(**{**cfg, 'samples_per_src': 500_000})
        else:
            raise
    return paths

# ── Sort by distance ──────────────────────────────────────────────────────────
_all_rx = list(receivers)
total   = len(_all_rx)
tx_pos2d = tx_pos[:2]
_all_rx.sort(key=lambda rx: float(np.linalg.norm(
    [_safe_ps(rx.position[0]) - tx_pos2d[0], _safe_ps(rx.position[1]) - tx_pos2d[1]])))

ts          = datetime.now().strftime('%Y%m%d_%H%M%S')
summary_csv = os.path.join(OUT_DIR, f'path_solver_summary_900s2_{ts}.csv')
per_ray_csv = os.path.join(OUT_DIR, f'path_solver_per_ray_900s2_{ts}.csv') if SAVE_PER_RAY else None

summary_rows = []
per_ray_rows = []
errors = 0
t0 = time.time()

print(f'\nProcessing {total} receivers in batches of {BATCH_SIZE} ...')

for b_start in range(0, total, BATCH_SIZE):
    batch    = _all_rx[b_start : b_start + BATCH_SIZE]
    max_dist = max(float(np.linalg.norm(
        [_safe_ps(rx.position[0]) - tx_pos2d[0], _safe_ps(rx.position[1]) - tx_pos2d[1]]))
        for rx in batch)
    n_samp = adaptive_samples(max_dist)
    cfg    = {**PS_CONFIG_BASE, 'samples_per_src': n_samp}

    paths = None
    batch_paths = 0
    try:
        paths       = run_batch(batch, cfg)
        a_all       = extract_amplitudes(paths)
        batch_paths = int(np.sum(np.abs(a_all) ** 2 > 1e-30))
    except Exception as _e:
        print(f'  [WARN] Batch {b_start}: {_e}')
        paths = None; a_all = None; batch_paths = 0

    if paths is None or batch_paths == 0:
        for rx in batch:
            los_d = float(np.linalg.norm(
                np.array([_safe_ps(rx.position[0]),
                          _safe_ps(rx.position[1]),
                          _safe_ps(rx.position[2])]) - tx_pos))
            summary_rows.append({
                'receiver': rx.name,
                'x_m': _safe_ps(rx.position[0]),
                'y_m': _safe_ps(rx.position[1]),
                'z_m': _safe_ps(rx.position[2]),
                'dist_from_tx_m': los_d,
                'num_samples_used': n_samp,
                'n_paths': 0,
                'rssi_best_dbm': np.nan,
                'rssi_incoherent_dbm': np.nan,
                'rssi_coherent_dbm': np.nan,
            })
        if paths is None:
            errors += len(batch)
        if paths is not None:
            del paths
        gc.collect()
        done = min(b_start + BATCH_SIZE, total)
        if done % max(BATCH_SIZE, total // 10) < BATCH_SIZE or done == total:
            print(f'  [{done}/{total}]  {time.time()-t0:.0f}s  (0 paths — NLOS/far)')
        continue

    n_b, n_p  = a_all.shape
    tau_all   = extract_tau(paths, n_b, n_p)

    for i, rx in enumerate(batch):
        rx_pos   = np.array([_safe_ps(rx.position[0]),
                             _safe_ps(rx.position[1]),
                             _safe_ps(rx.position[2])])
        los_d    = float(np.linalg.norm(rx_pos - tx_pos))
        idx      = i if i < n_b else n_b - 1
        best_pl, incoh_pl, coh_pl, n_valid = summary_metrics(a_all[idx])

        # Convert path gain to RSSI using rssi_from_path_gain()
        _rssi_best  = rssi_from_path_gain(10**(-best_pl /10)) if not np.isnan(best_pl)  else np.nan
        _rssi_incoh = rssi_from_path_gain(10**(-incoh_pl/10)) if not np.isnan(incoh_pl) else np.nan
        _rssi_coh   = rssi_from_path_gain(10**(-coh_pl  /10)) if not np.isnan(coh_pl)   else np.nan

        summary_rows.append({
            'receiver'            : rx.name,
            'x_m'                 : _safe_ps(rx.position[0]),
            'y_m'                 : _safe_ps(rx.position[1]),
            'z_m'                 : _safe_ps(rx.position[2]),
            'dist_from_tx_m'      : los_d,
            'num_samples_used'    : n_samp,
            'n_paths'             : n_valid,
            'rssi_best_dbm'       : _rssi_best,
            'rssi_incoherent_dbm' : _rssi_incoh,
            'rssi_coherent_dbm'   : _rssi_coh,
        })

        if SAVE_PER_RAY and n_valid > 0:
            a_row   = a_all[idx]
            pwr_row = np.abs(a_row) ** 2
            order   = np.argsort(pwr_row)[::-1]
            max_pwr = pwr_row[order[0]]
            strong_phase = np.angle(a_row[order[0]], deg=True)
            for rank, ray_i in enumerate(order[:MAX_RAYS_PER_RX]):
                ac      = a_row[ray_i]
                pwr_ray = float(pwr_row[ray_i])
                if pwr_ray <= 1e-30:
                    break
                phase   = float(np.angle(ac, deg=True))
                ph_diff = (phase - strong_phase + 180) % 360 - 180
                delay   = float(tau_all[idx, ray_i]) if idx < tau_all.shape[0] and ray_i < tau_all.shape[1] and not np.isnan(tau_all[idx, ray_i]) else np.nan
                plen    = delay * _C_local if not np.isnan(delay) else np.nan
                rtype   = ray_type_heuristic(plen, los_d, pwr_ray, max_pwr) \
                          if not np.isnan(plen) else 'UNKNOWN'
                per_ray_rows.append({
                    'receiver'       : rx.name,
                    'rank'           : rank,
                    'ray_type'       : rtype,
                    'power_linear'   : pwr_ray,
                    'path_loss_db'   : -10 * np.log10(pwr_ray),
                    'amplitude_real' : float(ac.real),
                    'amplitude_imag' : float(ac.imag),
                    'phase_deg'      : phase,
                    'phase_diff_deg' : ph_diff,
                    'constructive'   : 'STRONGEST' if rank == 0
                                       else ('CONSTRUCTIVE' if abs(ph_diff) < 90 else 'DESTRUCTIVE'),
                    'delay_s'        : delay,
                    'path_length_m'  : plen,
                })

    del paths, a_all, tau_all
    gc.collect()
    done = min(b_start + BATCH_SIZE, total)
    if done % max(BATCH_SIZE, total // 10) < BATCH_SIZE or done == total:
        elapsed = time.time() - t0
        eta     = (total - done) / max(done / max(elapsed, 1e-9), 1e-9)
        print(f'  [{done}/{total}]  {elapsed:.0f}s elapsed  ETA {eta/60:.1f} min', flush=True)

# Restore all receivers
for nm in list(scene.receivers.keys()): scene.remove(nm)
for rx in _all_rx: scene.add(rx)

# ── Save ──────────────────────────────────────────────────────────────────────
df_ps = pd.DataFrame(summary_rows)
df_ps.to_csv(summary_csv, index=False)
print(f'\n  Summary  -> {summary_csv}')

if SAVE_PER_RAY and per_ray_rows:
    df_ray = pd.DataFrame(per_ray_rows)
    df_ray.to_csv(per_ray_csv, index=False)
    print(f'  Per-ray  -> {per_ray_csv}  ({len(df_ray):,} rays)')

elapsed = time.time() - t0
valid   = df_ps[df_ps['n_paths'] > 0]
nan_rx  = df_ps[df_ps['n_paths'] == 0]
print(f'\n  Total time      : {elapsed:.1f}s  |  Errors: {errors}')
print(f'  Receivers solved: {len(valid)}/{total} ({100*len(valid)/max(total,1):.1f}%)')
print(f'  Zero-path (NaN) : {len(nan_rx)}')

for col, lbl in [('rssi_incoherent_dbm', 'RSSI incoher.'),
                 ('rssi_best_dbm',        'RSSI best')]:
    v = df_ps[col].dropna()
    if len(v):
        print(f'  {lbl:20s}: mean={v.mean():.1f}  std={v.std():.1f}  min={v.min():.1f}  max={v.max():.1f}')

import matplotlib.pyplot as plt
_df_plot = df_ps.copy()
_df_plot['dist_km'] = _df_plot['dist_from_tx_m'] / 1000
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
_v = _df_plot.dropna(subset=['rssi_incoherent_dbm'])
_pl_v = [-rssi_from_path_gain(0) + r + TX_CONDUCTED_DBM + RX_EXTRA_GAIN_DB
         if not np.isnan(r) else np.nan
         for r in _v['rssi_incoherent_dbm']]
# Compute path loss from RSSI: PL = TX_CONDUCTED - RSSI + RX_EXTRA + SITE_CORR
_pl_vals = TX_CONDUCTED_DBM - _v['rssi_incoherent_dbm'] + RX_EXTRA_GAIN_DB + SITE_CORRECTION_DB
axes[0].scatter(_v['dist_km'], _pl_vals, s=5, alpha=0.5, c='steelblue')
axes[0].set(xlabel='Distance (km)', ylabel='Path Loss (dB)', title='Incoherent PL vs Distance')
axes[0].grid(alpha=0.3)
axes[1].hist(_df_plot['rssi_incoherent_dbm'].dropna(), bins=40, color='coral', edgecolor='white', alpha=0.8)
axes[1].set(xlabel='RSSI (dBm)', ylabel='Count', title='RSSI Distribution')
axes[1].grid(alpha=0.3)
axes[2].scatter(_df_plot['dist_km'], _df_plot['n_paths'], s=5, alpha=0.4, c='seagreen')
axes[2].set(xlabel='Distance (km)', ylabel='Num paths', title='Paths vs Distance')
axes[2].set_yscale('symlog')
axes[2].grid(alpha=0.3)
plt.suptitle(f'Path Solver 900 MHz (Sionna 2.0) — {len(df_ps)} receivers', fontsize=12)
plt.tight_layout()
_p = os.path.join(OUT_DIR, 'cell7_path_solver_900mhz_s2.png')
plt.savefig(_p, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved → {_p}')

In [ ]:
# ====================================================================
# CELL 8 — CUMULATIVE DISTANCE ANALYSIS  (scattering ON vs OFF)
# ====================================================================
# For each max-distance threshold, takes ALL receivers from 0 up to
# that distance, runs PathSolver with/without scattering, reports
# Bias, RMSE, MAE, R² vs measured RSSI.
# ====================================================================
import gc, time
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print('=' * 70)
print('CELL 8 — CUMULATIVE DISTANCE ANALYSIS  (scattering ON vs OFF)')
print('=' * 70)

# ── Cumulative distance thresholds (m) ───────────────────────────────────────
THRESHOLDS = [100, 200, 300, 500, 750, 900, 1000, 1250, 1500, 1750, 2000, 2250, 2500, 3000]

# ── Load measurements ────────────────────────────────────────────────────────
_df_m8 = pd.read_csv(MEASUREMENT_CSV)
_name_col = [c for c in _df_m8.columns if 'name' in c.lower() or 'id' in c.lower()][0]
_rssi_col = [c for c in _df_m8.columns if 'measurement' in c.lower()
             or ('rssi' in c.lower() and 'dbm' in c.lower())
             or c.lower() == 'local_measurement_dbm'][0]

# ── Build receiver lookup: name → {rx, dist_m, measured_rssi} ────────────────
_safe8  = lambda v: float(v.item()) if hasattr(v, 'item') else float(v)
_tx8    = list(scene.transmitters.values())[0]
_tx8_xy = np.array([_safe8(_tx8.position[0]), _safe8(_tx8.position[1])])

_rx_lookup = {}
for _rx8 in receivers:
    _xy = np.array([_safe8(_rx8.position[0]), _safe8(_rx8.position[1])])
    _rx_lookup[_rx8.name] = {'rx': _rx8, 'dist_m': float(np.linalg.norm(_xy - _tx8_xy))}

for _, _row8 in _df_m8.iterrows():
    _n8 = str(_row8[_name_col])
    if _n8 in _rx_lookup:
        _rx_lookup[_n8]['measured_rssi'] = float(_row8[_rssi_col])

_valid_rx = {n: v for n, v in _rx_lookup.items() if 'measured_rssi' in v}
print(f'Receivers with measurements: {len(_valid_rx)}')

# ── PathSolver configs ────────────────────────────────────────────────────────
_cfg_on  = dict(max_depth=MAX_DEPTH, los=True, specular_reflection=True,
                diffraction=True, edge_diffraction=True,
                diffuse_reflection=True,  samples_per_src=NUM_SAMPLES_PS)
_cfg_off = dict(max_depth=MAX_DEPTH, los=True, specular_reflection=True,
                diffraction=True, edge_diffraction=True,
                diffuse_reflection=False, samples_per_src=NUM_SAMPLES_PS)

# ── Solve a batch, return name→rssi_dbm ──────────────────────────────────────
def _solve(rx_list, cfg):
    if not rx_list:
        return {}
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    for _r in rx_list: scene.add(_r)
    try:
        _p = PathSolver()(scene, **cfg)
    except Exception as _e:
        print(f'    solver error: {_e}')
        return {_r.name: np.nan for _r in rx_list}

    _a_raw = _p.a
    if isinstance(_a_raw, tuple):
        _a_np = ((_a_raw[0].numpy() if hasattr(_a_raw[0], 'numpy') else np.array(_a_raw[0])) +
                 1j*(_a_raw[1].numpy() if hasattr(_a_raw[1], 'numpy') else np.array(_a_raw[1])))
    else:
        _a_np = _a_raw.numpy() if hasattr(_a_raw, 'numpy') else np.array(_a_raw)

    _a_sq = np.squeeze(_a_np)
    if _a_sq.ndim == 1:
        _a_sq = _a_sq[np.newaxis, :]

    out = {}
    for _i, _r in enumerate(rx_list):
        try:
            _row = _a_sq[_i] if _a_sq.ndim > 1 else _a_sq
            _pwr = float(np.sum(np.abs(_row) ** 2))
            out[_r.name] = rssi_from_path_gain(_pwr) if _pwr > 1e-30 else np.nan
        except Exception:
            out[_r.name] = np.nan

    del _p; gc.collect()
    return out

def _metrics(sim, meas):
    mask = ~(np.isnan(sim) | np.isnan(meas))
    n = int(mask.sum())
    if n < 2:
        return n, np.nan, np.nan, np.nan, np.nan
    s, m = sim[mask], meas[mask]
    return (n,
            float(np.mean(s - m)),
            float(np.sqrt(mean_squared_error(m, s))),
            float(mean_absolute_error(m, s)),
            float(r2_score(m, s)))

# ── Run per threshold ─────────────────────────────────────────────────────────
results = []
print()

for thr in THRESHOLDS:
    label   = f'0–{thr}m'
    _subset = [v for v in _valid_rx.values() if v['dist_m'] <= thr]
    N       = len(_subset)

    if N == 0:
        print(f'{label:>10}: no receivers')
        results.append({'band': label, 'max_dist_m': thr, 'N': 0})
        continue

    _rx_objs  = [v['rx'] for v in _subset]
    _measured = np.array([v['measured_rssi'] for v in _subset])

    print(f'{label:>10}:  N={N:4d}  solving ON...', end=' ', flush=True)
    t0 = time.time()

    _sim_on  = np.array([_solve(_rx_objs, _cfg_on ).get(r.name, np.nan) for r in _rx_objs])

    print(f'OFF...', end=' ', flush=True)
    _sim_off = np.array([_solve(_rx_objs, _cfg_off).get(r.name, np.nan) for r in _rx_objs])

    n_on,  b_on,  rmse_on,  mae_on,  r2_on  = _metrics(_sim_on,  _measured)
    n_off, b_off, rmse_off, mae_off, r2_off = _metrics(_sim_off, _measured)

    print(f'{time.time()-t0:.0f}s')
    print(f'{"":>10}   SCAT ON   N={n_on:4d}  bias={b_on:+6.1f} dB  RMSE={rmse_on:5.1f}  MAE={mae_on:5.1f}  R²={r2_on:.3f}')
    print(f'{"":>10}   SCAT OFF  N={n_off:4d}  bias={b_off:+6.1f} dB  RMSE={rmse_off:5.1f}  MAE={mae_off:5.1f}  R²={r2_off:.3f}')
    print()

    results.append({
        'band': label, 'max_dist_m': thr, 'N': N,
        'on_n':    n_on,   'off_n':    n_off,
        'on_bias': b_on,   'off_bias': b_off,
        'on_rmse': rmse_on,'off_rmse': rmse_off,
        'on_mae':  mae_on, 'off_mae':  mae_off,
        'on_r2':   r2_on,  'off_r2':   r2_off,
    })

# Restore scene receivers
for _nm in list(scene.receivers.keys()): scene.remove(_nm)
for _rx in receivers: scene.add(_rx)

# ── Summary table ─────────────────────────────────────────────────────────────
_df8 = pd.DataFrame(results)
print()
print('=' * 95)
print(f'{"Band":>10}  {"N":>4}  {"── Scattering ON ──────────────────":^38}  {"── Scattering OFF ─────────────────":^38}')
print(f'{"":>10}  {"":>4}  {"N_valid":>7} {"Bias":>7} {"RMSE":>6} {"MAE":>6} {"R²":>6}  {"N_valid":>7} {"Bias":>7} {"RMSE":>6} {"MAE":>6} {"R²":>6}')
print('-' * 95)
for _, r in _df8.iterrows():
    if r['N'] == 0:
        print(f'{r["band"]:>10}  {"0":>4}  {"— no receivers —":^38}')
        continue
    print(f'{r["band"]:>10}  {int(r["N"]):>4}  '
          f'{int(r["on_n"]):>7} {r["on_bias"]:>+7.1f} {r["on_rmse"]:>6.1f} {r["on_mae"]:>6.1f} {r["on_r2"]:>6.3f}  '
          f'{int(r["off_n"]):>7} {r["off_bias"]:>+7.1f} {r["off_rmse"]:>6.1f} {r["off_mae"]:>6.1f} {r["off_r2"]:>6.3f}')
print('=' * 95)

_out8 = os.path.join(OUT_DIR, f'cumulative_band_analysis_{pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")}.csv')
_df8.to_csv(_out8, index=False)
print(f'\nSaved: {_out8}')


## Cell 8 — Compare vs Measurements

In [ ]:
# ====================================================================
# CELL 8 — SIM vs MEASUREMENTS  [900 MHz / Sionna 2.0]
# ====================================================================
import math, numpy as np, pandas as pd, matplotlib.pyplot as plt, os, glob

if not MEASUREMENT_CSV or not os.path.exists(MEASUREMENT_CSV):
    print('Set MEASUREMENT_CSV in Cell 1 to compare vs measurements.')
else:
    if 'df_ps' not in dir():
        _files = sorted(glob.glob(os.path.join(OUT_DIR, 'path_solver_summary_900s2_*.csv')))
        assert _files, f'No path solver CSV found in {OUT_DIR}\nRun Cell 7 first.'
        df_ps = pd.read_csv(_files[-1])
        print(f'Loaded: {_files[-1]}')

    df_sim = df_ps.rename(columns={
        'receiver'           : 'name',
        'rssi_incoherent_dbm': 'rssi_sim_dbm',
        'dist_from_tx_m'     : 'dist_m',
    })

    df_meas  = pd.read_csv(MEASUREMENT_CSV)
    df_merge = df_sim.merge(df_meas[['name', 'local_measurement_dBm']], on='name', how='inner')
    df_merge = df_merge.dropna(subset=['rssi_sim_dbm', 'local_measurement_dBm'])
    df_merge['err']     = df_merge['rssi_sim_dbm'] - df_merge['local_measurement_dBm']
    df_merge['dist_km'] = df_merge['dist_m'] / 1000

    bias = df_merge['err'].mean()
    rmse = math.sqrt((df_merge['err']**2).mean())

    print(f'Receivers compared : {len(df_merge)}')
    print(f'Bias (sim-meas)    : {bias:+.2f} dB')
    print(f'RMSE               : {rmse:.2f} dB')
    print()

    print(f'  {"Band":<12} {"N":>4}  {"Bias (dB)":>10}  {"RMSE (dB)":>10}  {"Mean paths":>11}')
    print(f'  {"-"*12} {"-"*4}  {"-"*10}  {"-"*10}  {"-"*11}')
    for lbl, d0, d1 in [("<300m",0,300),("300-700m",300,700),
                        ("700m-1.2km",700,1200),(">1.2km",1200,9999)]:
        s = df_merge[(df_merge['dist_m']>=d0) & (df_merge['dist_m']<d1)]
        if not len(s): continue
        e = s['err']
        p = s['n_paths'].mean() if 'n_paths' in s.columns else float('nan')
        print(f'  {lbl:<12} {len(s):>4}  {e.mean():>+10.1f}  {((e**2).mean()**0.5):>10.1f}  {p:>11.0f}')

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].scatter(df_merge['dist_km'], df_merge['err'], s=6, alpha=0.5, color='steelblue')
    axes[0].axhline(0, color='red', lw=1)
    axes[0].set_xlabel('Distance (km)')
    axes[0].set_ylabel('Error (dB)')
    axes[0].set_title(f'Error vs distance  (bias={bias:+.1f} dB, RMSE={rmse:.1f} dB)')
    axes[0].grid(alpha=0.3)

    axes[1].scatter(df_merge['local_measurement_dBm'], df_merge['rssi_sim_dbm'],
                    s=6, alpha=0.5, color='steelblue')
    _lo = min(df_merge['local_measurement_dBm'].min(), df_merge['rssi_sim_dbm'].min()) - 5
    _hi = max(df_merge['local_measurement_dBm'].max(), df_merge['rssi_sim_dbm'].max()) + 5
    axes[1].plot([_lo, _hi], [_lo, _hi], 'r--', lw=1)
    axes[1].set_xlabel('Measured RSSI (dBm)')
    axes[1].set_ylabel('Simulated RSSI (dBm)')
    axes[1].set_title('Sim vs Measured  (900 MHz / Sionna 2.0)')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    _p = os.path.join(OUT_DIR, 'rssi_compare_900mhz_s2.png')
    plt.savefig(_p, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot saved → {_p}')

In [ ]:
# ====================================================================
# CELL REPORT — FINAL FLAT TERRAIN EXPERIMENT REPORT
# ====================================================================
# Saves a full Markdown + CSV report of Phase 1 results.
# Run after CELL 8 completes (or manually paste results below).
# ====================================================================
import os, json
from datetime import datetime

os.makedirs('results', exist_ok=True)

# ── Experiment configuration ─────────────────────────────────────────
cfg = {
    "city":             "Nottingham, UK",
    "campaign":         "Ofcom sub-6GHz 2018",
    "frequency_MHz":    915.95,
    "tx_lat":           TX_LAT,
    "tx_lon":           TX_LON,
    "tx_agl_m":         TX_AGL_M,
    "tx_conducted_dbm": TX_CONDUCTED_DBM,
    "tx_antenna_gain":  TX_ANTENNA_GAIN_DBI,
    "eirp_dbm":         EIRP_DBM,
    "rx_agl_m":         RX_AGL_M,
    "rx_extra_gain_db": RX_EXTRA_GAIN_DB,
    "antenna_pattern":  ANTENNA_PATTERN,
    "max_depth":        MAX_DEPTH,
    "flat_terrain":     FLAT_TERRAIN,
    "num_rx":           NUM_RX,
    "scene_xml":        SCENE_XML,
    "materials": {
        "itu_brick":          {"scatter": 0.25, "er": 3.75, "sigma": 0.038},
        "itu_concrete":       {"scatter": 0.20, "er": 5.31, "sigma": 0.092},
        "itu_wet_ground":     {"scatter": 0.40, "er": 30.0, "sigma": 0.020},
        "itu_very_dry_ground":{"scatter": 0.30, "er": 2.80, "sigma": 0.000},
        "itu_metal":          {"scatter": 0.05, "er": 1.00, "sigma": 1e7},
    },
}

# ── Phase 1 results (paste from CELL 8 output) ───────────────────────
# Format: (band_m, N, bias_on, rmse_on, mae_on, r2_on,
#                     bias_off, rmse_off, mae_off, r2_off)
RESULTS = [
    (100,   8,  +0.4,  5.3,  4.6, -1.108,  +0.4,  5.3,  4.6, -1.108),
    (200,  17,  -3.4,  6.3,  5.8, -2.101,  -3.4,  6.3,  5.8, -2.101),
    (300,  26,  -4.8,  7.8,  6.5, -0.713,  -4.8,  7.8,  6.5, -0.714),
    (500,  44,  -5.7,  8.2,  6.9,  0.380,  -5.7,  8.2,  6.9,  0.378),
    (750,  67,  -1.4, 10.6,  9.1,  0.484,  -1.5, 10.7,  9.1,  0.477),
    (900,  78,  -2.4, 11.3,  9.7,  0.463,  -2.8, 11.7,  9.9,  0.426),
    (1000, 87,  -3.0, 11.6, 10.0,  0.503,  -3.5, 12.0, 10.4,  0.463),
    # Beyond 1000m excluded — scene boundary artefact (+8.4 dB bias at 1250m)
]

ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# ── Save CSV ──────────────────────────────────────────────────────────
import csv
csv_path = f"results/phase1_flat_results_{ts}.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["band_m","N",
                "bias_on","rmse_on","mae_on","r2_on",
                "bias_off","rmse_off","mae_off","r2_off"])
    for row in RESULTS:
        w.writerow(row)
print(f"CSV saved: {csv_path}")

# ── Save JSON config ──────────────────────────────────────────────────
json_path = f"results/phase1_config_{ts}.json"
with open(json_path, "w") as f:
    json.dump(cfg, f, indent=2)
print(f"Config saved: {json_path}")

# ── Save Markdown report ──────────────────────────────────────────────
md_path = f"results/phase1_report_{ts}.md"
with open(md_path, "w") as f:
    f.write(f"""# Sionna RT Phase 1 Report — Flat Terrain
**Generated:** {datetime.now().strftime("%Y-%m-%d %H:%M")}

## Experiment Configuration

| Parameter | Value |
|---|---|
| City | {cfg["city"]} |
| Campaign | {cfg["campaign"]} |
| Frequency | {cfg["frequency_MHz"]} MHz |
| TX position | lat={cfg["tx_lat"]}, lon={cfg["tx_lon"]} |
| TX height AGL | {cfg["tx_agl_m"]} m |
| TX conducted power | {cfg["tx_conducted_dbm"]} dBm |
| TX antenna gain | {cfg["tx_antenna_gain"]} dBi |
| EIRP | {cfg["eirp_dbm"]} dBm |
| RX height AGL | {cfg["rx_agl_m"]} m |
| RX system gain | {cfg["rx_extra_gain_db"]} dB |
| Antenna pattern | {cfg["antenna_pattern"]} (half-wave dipole) |
| Max depth | {cfg["max_depth"]} |
| Terrain | Flat (z=0) |
| Receivers | {cfg["num_rx"]} |

## ITU-R P.2040-2 Materials

| Material | εr | σ (S/m) | Scatter |
|---|---|---|---|
| itu_brick | 3.75 | 0.038 | 0.25 |
| itu_concrete | 5.31 | 0.092 | 0.20 |
| itu_wet_ground | 30.0 | 0.020 | 0.40 |
| itu_very_dry_ground | 2.80 | 0.000 | 0.30 |
| itu_metal | 1.00 | 1e7 | 0.05 |

## Results — Cumulative Distance Bands (Scattering ON vs OFF)

| Band | N | Bias ON | RMSE ON | MAE ON | R² ON | Bias OFF | RMSE OFF | MAE OFF | R² OFF |
|---|---|---|---|---|---|---|---|---|---|
""")
    for (b, n, bon, ron, mon, r2on, boff, roff, moff, r2off) in RESULTS:
        f.write(f"| 0–{b}m | {n} | {bon:+.1f} | {ron:.1f} | {mon:.1f} | {r2on:.3f} | {boff:+.1f} | {roff:.1f} | {moff:.1f} | {r2off:.3f} |
")

    f.write(f"""
## Key Findings

### Scattering Effect
Lambertian scattering (ITU-R P.2040-2 coefficients) shows measurable improvement
at ranges >750m where NLOS paths dominate:
- 0–900m: R² improves from 0.426 (OFF) → 0.463 (ON) (+0.037)
- 0–1000m: R² improves from 0.463 (OFF) → 0.503 (ON) (+0.040)

### Near-Range Bias (0–500m)
Consistent negative bias (−4 to −6 dB) at short range indicates systematic
under-prediction. TX and RX positions verified outside buildings (OSM check).
Likely caused by building mesh walls absorbing near-field diffraction paths
that are physically present but not captured by the ray tracer at MAX_DEPTH=3.

### Scene Boundary Artefact
Results beyond 1000m are **excluded** due to a scene boundary artefact:
- 0–1000m bias: −3.0 dB (slight under-prediction)
- 0–1250m bias: +8.4 dB (massive over-prediction)

The OSM building mesh thins out at ~1km from TX. Receivers beyond this distance
see artificially clear propagation paths in simulation while being NLOS in reality.
This is a 3D scene coverage limitation, not a physics or calibration issue.

### Valid Simulation Range
**0–1000m** (87 receivers) — best achievable with current scene geometry.

Best result: **0–1000m, Scattering ON: RMSE=11.6 dB, R²=0.503**

## Phase 2 — Next Steps

To improve beyond these results:
1. **Expand scene bbox** in scene builder to ensure buildings cover 2–3 km radius
2. **Add real terrain (EA LiDAR DTM)** — expected 2–5 dB RMSE improvement at >750m
3. **Rebuild Blender scene** with terrain-aligned building bases

## R² Formula (Excel)

```
=1 - SUMXMY2(sim_range, meas_range) / DEVSQ(meas_range)
```
""")

print(f"Report saved: {md_path}")
print()
print("=" * 65)
print("PHASE 1 FINAL RESULTS SUMMARY")
print("=" * 65)
print(f"{'Band':>8}  {'N':>4}  {'Bias ON':>8}  {'RMSE ON':>8}  {'R² ON':>7}  {'RMSE OFF':>9}  {'R² OFF':>8}")
print("-" * 65)
for (b, n, bon, ron, mon, r2on, boff, roff, moff, r2off) in RESULTS:
    flag = " ← best" if b == 1000 else ""
    print(f"0-{b:>4}m  {n:>4}  {bon:>+8.1f}  {ron:>8.1f}  {r2on:>7.3f}  {roff:>9.1f}  {r2off:>8.3f}{flag}")
print()
print("Scene boundary artefact at >1000m — results beyond excluded.")
print(f"Best: 0-1000m SCAT ON  RMSE=11.6 dB  R²=0.503")
